# Fixed GMIC ensemble on reserved development patients

Keep private. Attach the official RSNA competition, notebook 09 version 4, and the reviewed notebook 10 version recorded in gmic_visual_review.json. GPU required. Evaluate all five fixed external checkpoints on the existing pilot cohort only. This is an adapted transfer-feasibility study, not the final reliability experiment. No final patients, calibration, threshold selection, or density-effect testing. Patient predictions are private; only aggregate development metrics are logged.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import importlib.util
PROJECT = Path('/kaggle/working/mammo_project')
(PROJECT / 'mammo_reliability').mkdir(parents=True, exist_ok=True)


In [ ]:
# Audited source: __init__.py
(PROJECT / 'mammo_reliability' / '__init__.py').write_text('"""Research software. No clinical validation is implied."""\n\n__version__ = "0.1.0"\n')


In [ ]:
# Audited source: audit.py
(PROJECT / 'mammo_reliability' / 'audit.py').write_text('"""Validate original RSNA metadata without silently changing the cohort."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nBREAST_KEY = ["patient_id", "laterality"]\nSTANDARD_VIEWS = ["CC", "MLO"]\nREQUIRED = ["site_id", "patient_id", "image_id", "laterality", "view", "cancer", "density"]\nOUTCOME_ONLY = ["cancer", "biopsy", "invasive", "BIRADS", "difficult_negative_case"]\n\n\ndef require(condition: bool, message: str) -> None:\n    """Unlike assert, this check is not disabled by Python\'s -O option."""\n    if not bool(condition):\n        raise ValueError(message)\n\n\ndef file_sha256(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef environment() -> dict:\n    packages = {}\n    for name in ["numpy", "pandas", "scipy", "scikit-learn", "Pillow", "pydicom",\n                 "torch", "torchvision", "pylibjpeg", "pylibjpeg-openjpeg"]:\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {"python": platform.python_version(), "platform": platform.platform(), "packages": packages}\n\n\ndef write_json(path: str | Path, payload: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef locate_dataset() -> Path:\n    candidates = [Path("/kaggle/input/competitions/rsna-breast-cancer-detection"),\n                  Path("/kaggle/input/rsna-breast-cancer-detection")]\n    for root in candidates:\n        if (root / "train.csv").is_file():\n            return root\n    raise FileNotFoundError("Attach the official RSNA competition dataset or provide its local root explicitly.")\n\n\ndef audit_metadata(images: pd.DataFrame) -> tuple[pd.DataFrame, dict, dict[str, pd.DataFrame]]:\n    require(len(images) > 0, "Metadata is empty.")\n    require(set(REQUIRED).issubset(images.columns), f"Missing columns: {set(REQUIRED) - set(images.columns)}")\n    for col in ["site_id", "patient_id", "image_id", "laterality", "view"]:\n        require(images[col].notna().all(), f"Missing identifying information in {col}.")\n    for col in ["site_id", "patient_id", "image_id"]:\n        values = pd.to_numeric(images[col], errors="coerce")\n        require((values.notna() & np.isfinite(values) & (values >= 0) & (values == np.floor(values))).all(),\n                f"{col} must contain nonnegative integer identifiers.")\n    require(images["image_id"].is_unique, "Repeated image IDs require investigation.")\n    require(images["laterality"].isin(["L", "R"]).all(), "Unexpected laterality.")\n    require(images["cancer"].isin([0, 1]).all(), "Missing or nonbinary cancer labels.")\n    require((images.groupby("patient_id")["site_id"].nunique() == 1).all(), "A patient occurs at multiple sites.")\n    grouped = images.groupby(BREAST_KEY, sort=True)\n    require((grouped["cancer"].nunique(dropna=False) == 1).all(), "Cancer labels conflict within a breast.")\n    observed_density = images["density"].dropna()\n    require(observed_density.isin(list("ABCD")).all(), "Unexpected density category; inspect the source values.")\n\n    breasts = grouped.agg(site_id=("site_id", "first"), cancer=("cancer", "first"),\n                          n_images=("image_id", "size"), density=("density", "first"),\n                          n_density_labels=("density", "count"),\n                          n_density_values=("density", "nunique"))\n    views = pd.crosstab(index=[images[c] for c in BREAST_KEY], columns=images["view"])\n    views = views.reindex(index=breasts.index, columns=STANDARD_VIEWS, fill_value=0)\n    breasts["n_cc"] = views["CC"]\n    breasts["n_mlo"] = views["MLO"]\n    breasts["has_both_views"] = (breasts["n_cc"] > 0) & (breasts["n_mlo"] > 0)\n    breasts["density_missing"] = breasts["n_density_labels"] == 0\n    breasts["density_partly_missing"] = ((breasts["n_density_labels"] > 0) &\n                                           (breasts["n_density_labels"] < breasts["n_images"]))\n    breasts["density_conflict"] = breasts["n_density_values"] > 1\n    # The first available value is retained only if all available values agree.\n    breasts.loc[breasts["density_conflict"], "density"] = np.nan\n    breasts["density_group"] = breasts["density"].map({"A": "A/B", "B": "A/B", "C": "C/D", "D": "C/D"})\n    breasts["has_extra_images"] = breasts["n_images"] > 2\n    if "age" in images:\n        breasts["age"] = grouped["age"].first()\n        breasts["age_conflict"] = grouped["age"].nunique() > 1\n        breasts.loc[breasts["age_conflict"], "age"] = np.nan\n    breasts = breasts.reset_index()\n\n    extra = breasts.groupby(["site_id", "cancer"], as_index=False).agg(\n        breast_count=("cancer", "size"), breasts_with_extra_images=("has_extra_images", "sum"),\n        extra_image_fraction=("has_extra_images", "mean"))\n    density = breasts.groupby("site_id", as_index=False).agg(\n        breast_count=("cancer", "size"), all_missing=("density_missing", "sum"),\n        partly_missing=("density_partly_missing", "sum"), conflicts=("density_conflict", "sum"))\n    tables = {\n        "site_cancer": breasts.groupby(["site_id", "cancer"]).size().rename("breast_count").reset_index(),\n        "extra_images": extra, "density_availability": density,\n        "density_cancer": breasts.assign(density=breasts["density"].fillna("Missing/conflicting"))\n            .groupby(["site_id", "density", "cancer"]).size().rename("breast_count").reset_index(),\n        "image_views": images["view"].value_counts(dropna=False).rename_axis("view").rename("image_count").reset_index(),\n        "missing_columns": images.isna().sum().rename_axis("column").rename("missing_image_rows").reset_index(),\n    }\n    summary = {\n        "source": "computed_from_supplied_metadata", "image_rows": int(len(images)),\n        "patients": int(images["patient_id"].nunique()), "breasts": int(len(breasts)),\n        "positive_breasts": int(breasts["cancer"].sum()),\n        "positive_patients": int(breasts.groupby("patient_id")["cancer"].max().sum()),\n        "positive_breast_fraction": float(breasts["cancer"].mean()),\n        "breasts_missing_standard_view": int((~breasts["has_both_views"]).sum()),\n        "breasts_with_extra_images": int(breasts["has_extra_images"].sum()),\n        "density_conflicts": int(breasts["density_conflict"].sum()),\n        "pixel_duplicates_checked": False,\n    }\n    return breasts, summary, tables\n\n\ndef make_patient_folds(breasts: pd.DataFrame, n_splits: int = 5, seed: int = 20260909) -> pd.DataFrame:\n    require(breasts[BREAST_KEY].duplicated().sum() == 0, "Expected one row per breast.")\n    require(breasts["has_both_views"].all(), "Resolve breasts missing CC or MLO before defining the paired cohort.")\n    patients = breasts.groupby("patient_id", as_index=False).agg(site_id=("site_id", "first"),\n                                                               patient_cancer=("cancer", "max"))\n    strata = patients["site_id"].astype(str) + ":" + patients["patient_cancer"].astype(str)\n    require(strata.value_counts().min() >= n_splits, "Insufficient patients per site/cancer stratum for these folds.")\n    patients["outer_fold"] = -1\n    for fold, (_, test_idx) in enumerate(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(patients, strata)):\n        patients.loc[test_idx, "outer_fold"] = fold\n    require((patients["outer_fold"] >= 0).all(), "Some patients lack a fold.")\n    return patients\n\n\ndef partition_outer_fold(patients: pd.DataFrame, fold: int, seed: int = 20260909) -> pd.DataFrame:\n    require(patients["patient_id"].is_unique, "Split input must contain unique patients.")\n    require(fold in patients["outer_fold"].unique(), "Unknown outer fold.")\n    test = patients["outer_fold"] == fold\n    dev = patients.loc[~test].copy()\n    strata = dev["site_id"].astype(str) + ":" + dev["patient_cancer"].astype(str)\n    # Fractions of the complete cohort: fit 60%, tune 10%, calibrate 10%, test 20%.\n    fit_idx, held_idx = train_test_split(dev.index, test_size=0.25, stratify=strata, random_state=seed + fold)\n    held = dev.loc[held_idx]\n    held_strata = held["site_id"].astype(str) + ":" + held["patient_cancer"].astype(str)\n    tune_idx, cal_idx = train_test_split(held.index, test_size=0.5, stratify=held_strata, random_state=seed + 100 + fold)\n    result = patients.copy()\n    result["role"] = "test"\n    result.loc[fit_idx, "role"] = "fit"\n    result.loc[tune_idx, "role"] = "tune"\n    result.loc[cal_idx, "role"] = "calibrate"\n    require((result.loc[test, "role"] == "test").all(), "Outer test patients leaked into development.")\n    require((result.loc[~test, "role"] != "test").all(), "Development patient role was not assigned.")\n    for role, frame in result.groupby("role"):\n        require(set(frame["patient_cancer"].unique()) == {0, 1}, f"Role {role} lacks a class.")\n    return result\n')


In [ ]:
# Audited source: cache_parts.py
(PROJECT / 'mammo_reliability' / 'cache_parts.py').write_text('"""Build private lossless cache parts, then validate and mount them together.\n\nThe partition is only for storage. It does not change patient evaluation folds.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .audit import STANDARD_VIEWS, file_sha256, locate_dataset, require, write_json\nfrom .imaging import check_cross_patient_duplicates, prepare_images\n\n\ndef preprocessing_fingerprint() -> str:\n    return file_sha256(Path(__file__).with_name(\'imaging.py\'))\n\n\ndef _prepare_shard(task):\n    images, data_root, folder, height, width = task\n    return prepare_images(images, data_root, folder, height, width, True)\n\n\ndef build_part(data_root: Path, output: Path, part: int, n_parts: int = 2,\n               workers: int = 4, height: int = 1024, width: int = 768) -> dict:\n    require(n_parts > 0 and 0 <= part < n_parts and workers > 0, \'Invalid cache partition settings.\')\n    images = pd.read_csv(data_root / \'train.csv\')\n    images = images.loc[images[\'view\'].isin(STANDARD_VIEWS)].copy()\n    selected = images.loc[images[\'patient_id\'] % n_parts == part].copy()\n    require(len(selected) > 0, \'Empty cache part.\')\n    identity = {\'version\': 1, \'part\': part, \'n_parts\': n_parts, \'workers\': workers,\n                \'height\': height, \'width\': width, \'metadata_sha256\': file_sha256(data_root / \'train.csv\'),\n                \'image_processing_source_sha256\': preprocessing_fingerprint(),\n                \'n_images\': len(selected), \'cache_format\': \'16-bit PNG after full-frame letterbox\'}\n    output.mkdir(parents=True, exist_ok=True)\n    identity_file = output / \'part_identity.json\'\n    if identity_file.exists():\n        require(json.loads(identity_file.read_text()) == identity, \'Cache part settings or preprocessing changed.\')\n    else:\n        write_json(identity_file, identity)\n    tasks = []\n    shard_id = (selected[\'patient_id\'] // n_parts) % workers\n    for shard in range(workers):\n        subset = selected.loc[shard_id == shard]\n        if len(subset):\n            tasks.append((subset, data_root, output / f\'shard_{shard}\', height, width))\n    print(f\'Preparing private cache part {part + 1}/{n_parts}: {len(selected)} images; {len(tasks)} workers.\', flush=True)\n    if workers == 1:\n        frames = [_prepare_shard(task) for task in tasks]\n    else:\n        with ProcessPoolExecutor(max_workers=workers) as pool:\n            frames = list(pool.map(_prepare_shard, tasks))\n    manifest = pd.concat(frames, ignore_index=True).sort_values([\'patient_id\', \'image_id\'])\n    require(manifest[\'image_id\'].is_unique, \'Repeated image in cache part.\')\n    require(set(manifest[\'image_id\']) == set(selected[\'image_id\']), \'Cache part image coverage differs from metadata.\')\n    check_cross_patient_duplicates(manifest)\n    manifest[\'cache_relative_path\'] = manifest[\'cache_path\'].map(lambda p: str(Path(p).relative_to(output)))\n    manifest.to_csv(output / \'manifest.csv\', index=False)\n    result = {**identity, \'manifest_sha256\': file_sha256(output / \'manifest.csv\'),\n              \'cross_patient_exact_pixel_duplicates_within_part\': 0}\n    write_json(output / \'part_complete.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef assemble_parts(part_roots: list[Path], data_root: Path, output: Path,\n                   height: int = 1024, width: int = 768) -> pd.DataFrame:\n    require(len(part_roots) > 0, \'Attach the completed private cache parts first.\')\n    metadata_hash = file_sha256(data_root / \'train.csv\')\n    frames, completions = [], []\n    for root in part_roots:\n        completion = json.loads((root / \'part_complete.json\').read_text())\n        require(completion[\'metadata_sha256\'] == metadata_hash, \'A cache part uses different source metadata.\')\n        require(completion[\'height\'] == height and completion[\'width\'] == width, \'Cache part resolution mismatch.\')\n        require(completion[\'image_processing_source_sha256\'] == preprocessing_fingerprint(), \'Cache preprocessing source mismatch.\')\n        require(completion[\'manifest_sha256\'] == file_sha256(root / \'manifest.csv\'), \'A cache part manifest changed.\')\n        frame = pd.read_csv(root / \'manifest.csv\')\n        require(len(frame) == completion[\'n_images\'], \'Incomplete cache part.\')\n        require((frame[\'patient_id\'] % completion[\'n_parts\'] == completion[\'part\']).all(), \'Cache contains the wrong patients.\')\n        def remount(relative):\n            path = Path(relative)\n            require(not path.is_absolute() and \'..\' not in path.parts, \'Unsafe relative cache path.\')\n            target = root / path\n            require(target.is_file(), \'A cached image is missing from the mounted input.\')\n            return str(target)\n        frame[\'cache_path\'] = frame[\'cache_relative_path\'].map(remount)\n        frame[\'source_path\'] = [str(data_root / \'train_images\' / str(int(p)) / f\'{int(i)}.dcm\')\n                                for p, i in zip(frame[\'patient_id\'], frame[\'image_id\'])]\n        frames.append(frame)\n        completions.append(completion)\n    expected_parts = completions[0][\'n_parts\']\n    require(all(c[\'n_parts\'] == expected_parts for c in completions), \'Incompatible cache partition counts.\')\n    require(len(completions) == expected_parts and {c[\'part\'] for c in completions} == set(range(expected_parts)),\n            \'Duplicate or missing cache parts.\')\n    manifest = pd.concat(frames, ignore_index=True).sort_values([\'patient_id\', \'image_id\']).reset_index(drop=True)\n    images = pd.read_csv(data_root / \'train.csv\')\n    expected = images.loc[images[\'view\'].isin(STANDARD_VIEWS)]\n    require(manifest[\'image_id\'].is_unique and set(manifest[\'image_id\']) == set(expected[\'image_id\']),\n            \'Combined cache does not cover every standard-view image exactly once.\')\n    keys = [\'patient_id\', \'image_id\', \'laterality\', \'view\']\n    require(manifest[keys].sort_values(\'image_id\').reset_index(drop=True).equals(\n        expected[keys].sort_values(\'image_id\').reset_index(drop=True)), \'Cache image identifiers or views mismatch metadata.\')\n    check_cross_patient_duplicates(manifest)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(output / \'manifest.csv\', index=False)\n    write_json(output / \'preprocessing.json\', {\'version\': 1, \'height\': height, \'width\': width,\n               \'cache_images\': True, \'metadata_sha256\': metadata_hash})\n    write_json(output / \'pixel_audit_complete.json\', {\'metadata_sha256\': metadata_hash,\n               \'n_images\': len(manifest), \'manifest_sha256\': file_sha256(output / \'manifest.csv\'),\n               \'cross_patient_exact_pixel_duplicates\': 0, \'cache_part_completions\': completions})\n    print(f\'Validated and mounted {len(manifest)} images from {expected_parts} private parts.\', flush=True)\n    return manifest\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'stage\', choices=[\'build\', \'assemble\'])\n    parser.add_argument(\'--data-root\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--part\', type=int, default=0)\n    parser.add_argument(\'--n-parts\', type=int, default=2)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--height\', type=int, default=1024)\n    parser.add_argument(\'--width\', type=int, default=768)\n    parser.add_argument(\'--parts\', type=Path, nargs=\'+\')\n    args = parser.parse_args()\n    data_root = args.data_root or locate_dataset()\n    if args.stage == \'build\':\n        build_part(data_root, args.output, args.part, args.n_parts, args.workers, args.height, args.width)\n    else:\n        assemble_parts(args.parts or [], data_root, args.output, args.height, args.width)\n\n\nif __name__ == \'__main__\':\n    main()\n')


In [ ]:
# Audited source: evaluation.py
(PROJECT / 'mammo_reliability' / 'evaluation.py').write_text('"""Calibration and review-budget evaluation, with patient-cluster uncertainty."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport hashlib\nimport math\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score\n\nfrom .audit import BREAST_KEY, require\n\n\ndef checked_probability(p) -> np.ndarray:\n    p = np.asarray(p, dtype=float)\n    require(p.ndim == 1 and len(p) > 0, "Expected a nonempty probability vector.")\n    require(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), "Invalid probabilities.")\n    return p\n\n\ndef checked_labels(y, length: int) -> np.ndarray:\n    y = np.asarray(y)\n    require(y.ndim == 1 and len(y) == length and np.isin(y, [0, 1]).all(), "Expected matching binary labels.")\n    return y.astype(int)\n\n\n@dataclass\nclass MonotonePlatt:\n    """Positive-slope logistic calibration; fit only on calibration patients."""\n    slope: float = 1.0\n    intercept: float = 0.0\n\n    def fit(self, p, y):\n        p = checked_probability(p)\n        y = checked_labels(y, len(p))\n        require(len(np.unique(y)) == 2, "Calibration needs both classes.")\n        z = logit(np.clip(p, 1e-7, 1 - 1e-7))\n\n        def objective(params):\n            q = np.exp(params[0]) * z + params[1]\n            return float(np.mean(np.logaddexp(0, q) - y * q) + 1e-6 * np.sum(np.square(params)))\n\n        fitted = minimize(objective, [0.0, 0.0], method="L-BFGS-B", bounds=[(-4, 4), (-20, 20)])\n        require(fitted.success and np.isfinite(fitted.fun), f"Calibration did not converge: {fitted.message}")\n        self.slope, self.intercept = float(np.exp(fitted.x[0])), float(fitted.x[1])\n        return self\n\n    def predict(self, p):\n        p = checked_probability(p)\n        return expit(self.slope * logit(np.clip(p, 1e-7, 1 - 1e-7)) + self.intercept)\n\n\ndef sensitivity_threshold(p, y, target: float = 0.90) -> float:\n    p = checked_probability(p)\n    y = checked_labels(y, len(p))\n    require(0 < target <= 1, "Sensitivity target must be in (0, 1].")\n    positives = np.sort(p[y == 1])[::-1]\n    require(len(positives) > 0, "No positive calibration breasts.")\n    # Prediction is positive for p >= threshold. Tied scores are all positive.\n    return float(positives[math.ceil(target * len(positives)) - 1])\n\n\ndef classification_metrics(y, p, threshold: float) -> dict:\n    p = checked_probability(p)\n    y = checked_labels(y, len(p))\n    require(np.isfinite(threshold) and 0 <= threshold <= 1, "Invalid fixed threshold.")\n    positive = p >= threshold\n    tp, fn = int(((y == 1) & positive).sum()), int(((y == 1) & ~positive).sum())\n    tn, fp = int(((y == 0) & ~positive).sum()), int(((y == 0) & positive).sum())\n    safe_ratio = lambda a, b: float(a / b) if b else None\n    bins = np.minimum((p * 10).astype(int), 9)\n    ece = sum(float((bins == b).mean() * abs(p[bins == b].mean() - y[bins == b].mean()))\n              for b in range(10) if (bins == b).any())\n    both = len(np.unique(y)) == 2\n    return {"n_breasts": len(y), "n_positive": int(y.sum()), "tp": tp, "fn": fn, "tn": tn, "fp": fp,\n            "sensitivity": safe_ratio(tp, tp + fn), "specificity": safe_ratio(tn, tn + fp),\n            "ppv_in_study_sample": safe_ratio(tp, tp + fp), "npv_in_study_sample": safe_ratio(tn, tn + fn),\n            "roc_auc": float(roc_auc_score(y, p)) if both else None,\n            "average_precision": float(average_precision_score(y, p)) if y.sum() else None,\n            "brier": float(brier_score_loss(y, p)), "log_loss": float(log_loss(y, p, labels=[0, 1])),\n            "ece_10_equal_width_bins": ece, "threshold": float(threshold)}\n\n\ndef ensemble_features(predictions: pd.DataFrame, expected_seeds: list[int]) -> pd.DataFrame:\n    keys = BREAST_KEY + ["seed"]\n    required = set(keys + ["z_cc", "z_mlo"])\n    require(required.issubset(predictions), "Prediction schema is incomplete.")\n    require(not predictions[keys].duplicated().any(), "Duplicate breast/member predictions.")\n    require(len(expected_seeds) >= 2 and len(set(expected_seeds)) == len(expected_seeds), "Ensemble needs distinct members.")\n    require(set(predictions["seed"].unique()) == set(expected_seeds), "Unexpected or missing ensemble seed.")\n    require((predictions.groupby(BREAST_KEY)["seed"].nunique() == len(expected_seeds)).all(), "Incomplete ensemble for a breast.")\n    require(np.isfinite(predictions[["z_cc", "z_mlo"]].to_numpy()).all(), "Nonfinite model predictions.")\n    tmp = predictions.copy()\n    tmp["p_member"] = expit((tmp["z_cc"] + tmp["z_mlo"]) / 2)\n    tmp["p_cc"] = expit(tmp["z_cc"])\n    tmp["p_mlo"] = expit(tmp["z_mlo"])\n    group = tmp.groupby(BREAST_KEY)\n    result = group.agg(p_raw=("p_member", "mean"), p_cc=("p_cc", "mean"), p_mlo=("p_mlo", "mean"))\n    result["ensemble_variance"] = group["p_member"].var(ddof=0)\n    result["view_disagreement"] = abs(result["p_cc"] - result["p_mlo"])\n    return result.reset_index()\n\n\ndef reference_percentile(reference, values) -> np.ndarray:\n    """Midrank empirical CDF learned from calibration negatives only."""\n    ref = np.sort(np.asarray(reference, dtype=float))\n    values = np.asarray(values, dtype=float)\n    require(len(ref) > 0 and np.isfinite(ref).all() and np.isfinite(values).all(), "Invalid percentile reference.")\n    return (np.searchsorted(ref, values, side="left") + np.searchsorted(ref, values, side="right")) / (2 * len(ref))\n\n\ndef add_review_scores(calibration: pd.DataFrame, target: pd.DataFrame, threshold: float) -> pd.DataFrame:\n    reference = calibration.loc[calibration["p_calibrated"] < threshold]\n    require(len(reference) > 0, "No calibration negatives for score normalization.")\n    result = target.copy()\n    result["score_confidence"] = result["p_calibrated"]\n    result["score_disagreement"] = result["view_disagreement"]\n    result["score_uncertainty"] = result["ensemble_variance"]\n    normalized = []\n    for col in ["p_calibrated", "view_disagreement", "ensemble_variance"]:\n        normalized.append(reference_percentile(reference[col], result[col]))\n    result["score_confidence_disagreement"] = (normalized[0] + normalized[1]) / 2\n    result["score_confidence_uncertainty"] = (normalized[0] + normalized[2]) / 2\n    result["score_combined"] = sum(normalized) / 3\n    result["predicted_positive"] = result["p_calibrated"] >= threshold\n    return result\n\n\ndef review_selection(frame: pd.DataFrame, score_column: str, budget: float = 0.05, seed: int = 20260909) -> pd.Series:\n    require(0 <= budget <= 1, "Review budget must be between zero and one.")\n    require(frame.index.is_unique, "Reset the dataframe index before selecting review candidates.")\n    require(not frame[BREAST_KEY].duplicated().any(), "Review input must contain unique breasts.")\n    require(frame["predicted_positive"].isin([True, False]).all(), "Missing decision flags.")\n    checked_labels(frame["cancer"], len(frame))\n    require(np.isfinite(frame[score_column]).all(), "Nonfinite review scores.")\n    negative = frame.loc[~frame["predicted_positive"].astype(bool)].copy()\n    # Tie ordering uses only identifiers and a fixed seed, never labels or table order.\n    negative["tie"] = [hashlib.sha256(f"{seed}:{p}:{l}".encode()).hexdigest()\n                       for p, l in negative[BREAST_KEY].itertuples(index=False, name=None)]\n    negative = negative.sort_values([score_column, "tie"], ascending=[False, True])\n    k = math.floor(budget * len(negative))\n    selection = pd.Series(False, index=frame.index)\n    selection.loc[negative.iloc[:k].index] = True\n    return selection\n\n\ndef review_at_budget(frame: pd.DataFrame, score_column: str, budget: float = 0.05, seed: int = 20260909) -> dict:\n    selected = review_selection(frame, score_column, budget, seed)\n    negative = frame.loc[~frame["predicted_positive"].astype(bool)]\n    k = int(selected.sum())\n    fn = int(negative["cancer"].sum())\n    caught = int(frame.loc[selected, "cancer"].sum())\n    return {"method": score_column, "budget": budget, "negative_breasts": len(negative),\n            "reviewed_breasts": k, "false_negatives": fn, "false_negatives_flagged": caught,\n            "fn_capture_fraction": float(caught / fn) if fn else None,\n            "review_yield": float(caught / k) if k else None,\n            "actual_review_fraction": float(k / len(negative)) if len(negative) else 0.0,\n            "random_expected_fn_capture": float(k / len(negative)) if len(negative) and fn else None,\n            "interpretation": "flagging potential review candidates; no human review or cancer recovery was measured"}\n\n\ndef paired_cluster_bootstrap(frame: pd.DataFrame, method: str = "score_combined", baseline: str = "score_confidence",\n                             budget: float = 0.05, repeats: int = 2000, seed: int = 20260909) -> dict:\n    """Patient bootstrap of pooled capture difference, allocating budget per fold.\n\n    Multiplicity weights represent sampled patient clusters, including both breasts.\n    All fitted scores/thresholds stay fixed. A replicate with zero total false\n    negatives is undefined. This does not include model-retraining uncertainty.\n    """\n    require(repeats > 0, "Bootstrap repetitions must be positive.")\n    require(0 <= budget <= 1, "Review budget must be between zero and one.")\n    require(not frame[BREAST_KEY].duplicated().any(), "Expected unique held-out breasts.")\n    require(frame.groupby("patient_id")["outer_fold"].nunique().max() == 1, "Patient spans held-out folds.")\n    rng = np.random.default_rng(seed)\n    fold_groups = []\n    for _, fold in frame.groupby("outer_fold"):\n        # Validation and point-estimate routines use the same fixed tie ordering.\n        review_selection(fold, method, budget, seed)\n        review_selection(fold, baseline, budget, seed)\n        patients = np.sort(fold["patient_id"].unique())\n        negative = fold.loc[~fold["predicted_positive"].astype(bool)]\n        group_idx = np.searchsorted(patients, negative["patient_id"].to_numpy())\n        labels = negative["cancer"].to_numpy(dtype=int)\n        ties = np.array([hashlib.sha256(f"{seed}:{p}:{l}".encode()).hexdigest()\n                         for p, l in negative[BREAST_KEY].itertuples(index=False, name=None)])\n        orders = [np.lexsort((ties, -negative[col].to_numpy(dtype=float))) for col in [method, baseline]]\n        fold_groups.append((len(patients), group_idx, labels, orders))\n    estimates, undefined = [], 0\n    for _ in range(repeats):\n        total_fn, total_difference = 0, 0\n        for n_patients, group_idx, labels, orders in fold_groups:\n            multiplicity = rng.multinomial(n_patients, np.full(n_patients, 1 / n_patients))\n            weight = multiplicity[group_idx]\n            total_fn += int(weight @ labels)\n            k = math.floor(budget * weight.sum())\n            caught = []\n            for order in orders:\n                ordered_weight = weight[order]\n                preceding = np.cumsum(ordered_weight) - ordered_weight\n                selected_weight = np.minimum(ordered_weight, np.maximum(0, k - preceding))\n                caught.append(int(selected_weight @ labels[order]))\n            total_difference += caught[0] - caught[1]\n        if total_fn == 0:\n            undefined += 1\n        else:\n            estimates.append(float(total_difference / total_fn))\n    point_difference, point_fn = 0, 0\n    for _, fold in frame.groupby("outer_fold"):\n        a = review_at_budget(fold, method, budget, seed)\n        b = review_at_budget(fold, baseline, budget, seed)\n        point_difference += a["false_negatives_flagged"] - b["false_negatives_flagged"]\n        point_fn += a["false_negatives"]\n    result = {"method": method, "baseline": baseline, "budget": budget,\n              "estimand": "pooled FN capture difference with the review budget applied separately in each held-out fold",\n              "point_estimate": float(point_difference / point_fn) if point_fn else None,\n              "valid_bootstrap_replicates": len(estimates), "undefined_replicates": undefined,\n              "requested_replicates": repeats, "ci95": None,\n              "uncertainty_scope": "conditional on fitted models, calibration, and threshold; not model retraining"}\n    if len(estimates) >= max(20, math.ceil(repeats * 0.8)) and result["point_estimate"] is not None:\n        result["ci95"] = [float(x) for x in np.quantile(estimates, [0.025, 0.975])]\n    return result\n')


In [ ]:
# Audited source: imaging.py
(PROJECT / 'mammo_reliability' / 'imaging.py').write_text('"""DICOM presentation processing, preserving the full field of view."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport pydicom\nfrom pydicom.pixels import apply_modality_lut, apply_voi_lut\n\nfrom .audit import STANDARD_VIEWS, file_sha256, require, write_json\n\n\ndef normalize_pixels(raw: np.ndarray, presented: np.ndarray, photometric: str,\n                     padding_value=None, padding_limit=None) -> np.ndarray:\n    require(raw.ndim == 2 and presented.shape == raw.shape, "Only two-dimensional grayscale images are supported.")\n    require(photometric in ["MONOCHROME1", "MONOCHROME2"], "Unsupported photometric interpretation.")\n    valid = np.isfinite(presented)\n    if padding_value is not None:\n        if padding_limit is None:\n            valid &= raw != padding_value\n        else:\n            lower, upper = sorted([padding_value, padding_limit])\n            valid &= ~((raw >= lower) & (raw <= upper))\n    require(valid.any(), "No valid image pixels remain.")\n    low, high = float(presented[valid].min()), float(presented[valid].max())\n    require(high > low, "Uniform image after presentation processing.")\n    result = ((presented.astype(np.float32) - low) / (high - low)).clip(0, 1)\n    if photometric == "MONOCHROME1":\n        result = 1 - result\n    result[~valid] = 0\n    return result.astype(np.float32)\n\n\ndef letterbox(array: np.ndarray, height: int = 1024, width: int = 768) -> np.ndarray:\n    require(array.ndim == 2 and min(height, width) > 0, "Invalid image or target dimensions.")\n    h, w = array.shape\n    factor = min(height / h, width / w)\n    new_h, new_w = max(1, round(h * factor)), max(1, round(w * factor))\n    resized = np.asarray(Image.fromarray(array.astype(np.float32)).resize(\n        (new_w, new_h), resample=Image.Resampling.BILINEAR))\n    canvas = np.zeros((height, width), dtype=np.float32)\n    y, x = (height - new_h) // 2, (width - new_w) // 2\n    canvas[y:y + new_h, x:x + new_w] = resized\n    return canvas\n\n\ndef decode_dicom(path: str | Path, height: int = 1024, width: int = 768) -> tuple[np.ndarray, dict]:\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim == 3 and raw.shape[0] == 1:\n        raw = raw[0]\n    require(raw.ndim == 2, "Multiframe/color DICOM requires a separate preprocessing decision.")\n    require("PresentationLUTSequence" not in ds, "Presentation LUT sequence requires image-specific review.")\n    photo = str(ds.get("PhotometricInterpretation", ""))\n    presented = apply_voi_lut(apply_modality_lut(raw, ds), ds)\n    normalized = normalize_pixels(raw, presented, photo, ds.get("PixelPaddingValue"), ds.get("PixelPaddingRangeLimit"))\n    digest = hashlib.sha256()\n    digest.update(str((raw.shape, str(raw.dtype))).encode())\n    digest.update(np.ascontiguousarray(raw).tobytes())\n    info = {"height_original": int(raw.shape[0]), "width_original": int(raw.shape[1]),\n            "photometric": photo, "decoded_pixel_sha256": digest.hexdigest(),\n            "voi_present": bool("VOILUTSequence" in ds or "WindowCenter" in ds),\n            "nonzero_fraction_after_normalization": float((normalized > 0).mean())}\n    return letterbox(normalized, height, width), info\n\n\ndef check_cross_patient_duplicates(manifest: pd.DataFrame) -> None:\n    require("decoded_pixel_sha256" in manifest, "Pixel duplicate audit was not performed.")\n    shared = manifest.groupby("decoded_pixel_sha256")["patient_id"].nunique()\n    require(not (shared > 1).any(), "Identical decoded images occur under different patients. Investigate/group them before training.")\n\n\ndef prepare_images(images: pd.DataFrame, data_root: str | Path, output: str | Path,\n                   height: int = 1024, width: int = 768, cache_images: bool = True) -> pd.DataFrame:\n    """Resume verified 16-bit PNG conversion, or audit pixels without caching.\n\n    Never skip failed images. A stopped run can resume from the last manifest.\n    An explicit disk check after the first 32 images prevents filling the volume.\n    """\n    data_root, output = Path(data_root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    source_hash = file_sha256(data_root / "train.csv")\n    config = {"version": 1, "height": height, "width": width, "cache_images": cache_images, "metadata_sha256": source_hash}\n    config_file, manifest_file = output / "preprocessing.json", output / "manifest.csv"\n    if config_file.exists():\n        require(json.loads(config_file.read_text()) == config, "Preprocessing settings/source changed; use a new output directory.")\n    else:\n        write_json(config_file, config)\n    rows = images.loc[images["view"].isin(STANDARD_VIEWS)].sort_values(["patient_id", "image_id"])\n    previous = pd.read_csv(manifest_file).to_dict("records") if manifest_file.exists() else []\n    recorded = {int(r["image_id"]): r for r in previous}\n    require(len(recorded) == len(previous), "Repeated image in preprocessing manifest.")\n    require(set(recorded).issubset(set(rows["image_id"].astype(int))), "Manifest has unexpected images.")\n    cached_sizes = []\n    for row in rows.itertuples(index=False):\n        image_id, patient_id = int(row.image_id), int(row.patient_id)\n        if image_id in recorded:\n            old = recorded[image_id]\n            if cache_images:\n                require(Path(old["cache_path"]).is_file(), "A recorded cached image is missing.")\n                require(file_sha256(old["cache_path"]) == old["cache_sha256"], "Cached image hash mismatch.")\n                cached_sizes.append(Path(old["cache_path"]).stat().st_size)\n            continue\n        source = data_root / "train_images" / str(patient_id) / f"{image_id}.dcm"\n        array, info = decode_dicom(source, height, width)\n        record = {"patient_id": patient_id, "image_id": image_id, "laterality": row.laterality,\n                  "view": row.view, "source_path": str(source), **info}\n        if cache_images:\n            cache_path = output / "images" / str(patient_id) / f"{image_id}.png"\n            cache_path.parent.mkdir(parents=True, exist_ok=True)\n            Image.fromarray(np.round(array * 65535).astype(np.uint16)).save(cache_path)\n            record.update(cache_path=str(cache_path), cache_sha256=file_sha256(cache_path))\n            cached_sizes.append(cache_path.stat().st_size)\n        recorded[image_id] = record\n        if len(recorded) % 32 == 0 or len(recorded) == len(rows):\n            temporary = manifest_file.with_suffix(".csv.tmp")\n            pd.DataFrame(recorded.values()).to_csv(temporary, index=False)\n            temporary.replace(manifest_file)\n            print(f"Pixel audit: {len(recorded)}/{len(rows)} images", flush=True)\n            if cache_images:\n                estimate = float(np.mean(cached_sizes)) * (len(rows) - len(recorded)) * 1.25\n                require(estimate + 2 * 1024**3 < shutil.disk_usage(output).free,\n                        "Estimated PNG cache exceeds free disk space. Use a larger volume, or a NEW output directory with cache_images=False.")\n    manifest = pd.DataFrame(recorded.values()).sort_values(["patient_id", "image_id"]).reset_index(drop=True)\n    require(len(manifest) == len(rows), "Incomplete image manifest.")\n    check_cross_patient_duplicates(manifest)\n    write_json(output / "pixel_audit_complete.json", {"metadata_sha256": source_hash, "n_images": len(manifest),\n                                                      "manifest_sha256": file_sha256(manifest_file),\n                                                      "cross_patient_exact_pixel_duplicates": 0})\n    return manifest\n')


In [ ]:
# Audited source: model.py
(PROJECT / 'mammo_reliability' / 'model.py').write_text('"""Shared image encoder with equal CC/MLO weighting and breast-level loss."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport hashlib\nimport io\nimport random\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset\nfrom torchvision.models import ResNet18_Weights, resnet18\n\nfrom .audit import BREAST_KEY, require\nfrom .imaging import decode_dicom\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    # Some GPU reductions may still be nondeterministic; record warnings/environment.\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\nclass BreastDataset(Dataset):\n    def __init__(self, manifest: pd.DataFrame, breasts: pd.DataFrame, augment: bool = False,\n                 height: int = 1024, width: int = 768):\n        self.breasts = breasts.reset_index(drop=True).copy()\n        require(not self.breasts[BREAST_KEY].duplicated().any(), "Duplicate breasts in dataset.")\n        self.groups = {key: g.sort_values("image_id") for key, g in manifest.groupby(BREAST_KEY)}\n        require(set(map(tuple, self.breasts[BREAST_KEY].to_numpy())).issubset(self.groups), "A breast has no images.")\n        self.augment, self.height, self.width = augment, height, width\n\n    def __len__(self):\n        return len(self.breasts)\n\n    def __getitem__(self, index):\n        breast = self.breasts.iloc[index]\n        rows = self.groups[(breast.patient_id, breast.laterality)]\n        require(set(rows["view"].unique()) == {"CC", "MLO"}, "Expected both standard views.")\n        images, views = [], []\n        for row in rows.itertuples(index=False):\n            if hasattr(row, "cache_path") and pd.notna(row.cache_path):\n                payload = Path(row.cache_path).read_bytes()\n                require(hashlib.sha256(payload).hexdigest() == row.cache_sha256, "Cached pixels changed after the image audit.")\n                array = np.asarray(Image.open(io.BytesIO(payload)), dtype=np.float32) / 65535.0\n            else:\n                array, _ = decode_dicom(row.source_path, self.height, self.width)\n            require(array.shape == (self.height, self.width), "Cache shape does not match model configuration.")\n            # Full image is retained. No random crop or vertical flip.\n            if self.augment and random.random() < 0.5:\n                array = np.fliplr(array)\n            tensor = torch.from_numpy(np.ascontiguousarray(array)).unsqueeze(0).repeat(3, 1, 1)\n            tensor = (tensor - torch.tensor([0.485, 0.456, 0.406])[:, None, None]) / torch.tensor([0.229, 0.224, 0.225])[:, None, None]\n            images.append(tensor)\n            views.append(0 if row.view == "CC" else 1)\n        return {"images": torch.stack(images), "views": torch.tensor(views, dtype=torch.long),\n                "label": float(breast.cancer), "patient_id": int(breast.patient_id), "laterality": breast.laterality}\n\n\ndef collate_breasts(batch):\n    slots = [item["views"] + 2 * index for index, item in enumerate(batch)]\n    return {"images": torch.cat([item["images"] for item in batch]), "slots": torch.cat(slots),\n            "labels": torch.tensor([item["label"] for item in batch], dtype=torch.float32),\n            "patient_id": [item["patient_id"] for item in batch],\n            "laterality": [item["laterality"] for item in batch]}\n\n\ndef pool_view_logits(image_logits: torch.Tensor, slots: torch.Tensor, n_breasts: int) -> torch.Tensor:\n    require(image_logits.ndim == 1 and len(image_logits) == len(slots), "Logit/group shape mismatch.")\n    counts = torch.bincount(slots, minlength=2 * n_breasts)\n    require(len(counts) == 2 * n_breasts and (counts > 0).all().item(), "A breast lacks a standard view.")\n    sums = torch.zeros(2 * n_breasts, dtype=image_logits.dtype, device=image_logits.device)\n    sums = sums.index_add(0, slots, image_logits)\n    return (sums / counts.to(image_logits.dtype)).reshape(n_breasts, 2)\n\n\nclass MammographyModel(nn.Module):\n    def __init__(self, pretrained: bool = True):\n        super().__init__()\n        self.encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)\n        self.encoder.fc = nn.Linear(self.encoder.fc.in_features, 1)\n\n    def train(self, mode: bool = True):\n        super().train(mode)\n        # Freeze running BN statistics for small batches; affine parameters remain trainable.\n        for module in self.modules():\n            if isinstance(module, nn.BatchNorm2d):\n                module.eval()\n        return self\n\n    def forward(self, images, slots, n_breasts):\n        image_logits = self.encoder(images).squeeze(1)\n        view_logits = pool_view_logits(image_logits.float(), slots, n_breasts)\n        # Extra CC images do not give CC more weight than MLO, or this breast more weight in the loss.\n        return view_logits.mean(dim=1), view_logits\n')


In [ ]:
# Audited source: pilot.py
(PROJECT / 'mammo_reliability' / 'pilot.py').write_text('"""Development-only image preparation and model trials with explicit provenance."""\nfrom __future__ import annotations\n\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\n\nimport pandas as pd\n\nfrom .audit import (STANDARD_VIEWS, audit_metadata, file_sha256, locate_dataset,\n                    require, write_json)\nfrom .cache_parts import _prepare_shard, preprocessing_fingerprint\nfrom .imaging import check_cross_patient_duplicates\nfrom .study import make_study_patients, pilot_roles\n\n\ndef prepare_pilot_cache(data_root: Path, output: Path, config: dict,\n                        height: int = 1536, width: int = 1152, workers: int = 4) -> dict:\n    images = pd.read_csv(data_root / \'train.csv\')\n    breasts, _, _ = audit_metadata(images)\n    patients = make_study_patients(breasts, config[\'n_outer_folds\'], config[\'split_seed\'],\n                                   config[\'pilot_fraction\'], config[\'pilot_tune_fraction\'])\n    roles = pilot_roles(patients)\n    selected = images.loc[images.patient_id.isin(roles.patient_id) & images.view.isin(STANDARD_VIEWS)].copy()\n    require(workers > 0 and min(height, width) > 0, \'Invalid pilot cache settings.\')\n    identity = {\'purpose\': \'development_only\', \'metadata_sha256\': file_sha256(data_root / \'train.csv\'),\n                \'height\': height, \'width\': width, \'workers\': workers, \'n_images\': len(selected),\n                \'image_processing_source_sha256\': preprocessing_fingerprint(),\n                \'roles_sha256\': hashlib.sha256(roles.to_csv(index=False).encode()).hexdigest()}\n    output.mkdir(parents=True, exist_ok=True)\n    path = output / \'pilot_cache_identity.json\'\n    if path.exists():\n        require(json.loads(path.read_text()) == identity, \'Pilot cache settings changed.\')\n    else:\n        write_json(path, identity)\n    roles.to_csv(output / \'roles_private.csv\', index=False)\n    roles.groupby([\'role\', \'site_id\', \'patient_cancer\']).size().rename(\'patients\').to_csv(\n        output / \'partition_counts.csv\')\n    selected[\'worker\'] = selected.patient_id % workers\n    tasks = [(g.drop(columns=\'worker\'), data_root, output / f\'shard_{worker}\', height, width)\n             for worker, g in selected.groupby(\'worker\')]\n    print(f\'Preparing development-only {height} x {width} cache: {len(selected)} images, {len(roles)} patients.\', flush=True)\n    if workers == 1:\n        frames = [_prepare_shard(t) for t in tasks]\n    else:\n        with ProcessPoolExecutor(max_workers=workers) as pool:\n            frames = list(pool.map(_prepare_shard, tasks))\n    manifest = pd.concat(frames, ignore_index=True).sort_values(\'image_id\').reset_index(drop=True)\n    require(manifest.image_id.is_unique and set(manifest.image_id) == set(selected.image_id), \'Pilot cache coverage mismatch.\')\n    check_cross_patient_duplicates(manifest)\n    manifest[\'cache_relative_path\'] = manifest.cache_path.map(lambda p: str(Path(p).relative_to(output)))\n    manifest.to_csv(output / \'manifest.csv\', index=False)\n    result = {**identity, \'manifest_sha256\': file_sha256(output / \'manifest.csv\')}\n    write_json(output / \'pilot_cache_complete.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef load_pilot_cache(root: Path, reference: pd.DataFrame, roles: pd.DataFrame,\n                     model_config: dict, metadata_sha256: str) -> pd.DataFrame:\n    complete = json.loads((root / \'pilot_cache_complete.json\').read_text())\n    require(complete[\'purpose\'] == \'development_only\' and complete[\'metadata_sha256\'] == metadata_sha256,\n            \'Pilot cache uses another dataset or purpose.\')\n    require(complete[\'height\'] == model_config[\'height\'] and complete[\'width\'] == model_config[\'width\'],\n            \'Pilot cache resolution differs from the trial.\')\n    require(complete[\'image_processing_source_sha256\'] == preprocessing_fingerprint(), \'Pilot preprocessing changed.\')\n    require(complete[\'manifest_sha256\'] == file_sha256(root / \'manifest.csv\'), \'Pilot cache manifest changed.\')\n    require(complete[\'roles_sha256\'] == hashlib.sha256(roles.to_csv(index=False).encode()).hexdigest(),\n            \'Pilot cache patient assignments differ from this study.\')\n    manifest = pd.read_csv(root / \'manifest.csv\')\n    expected = reference.loc[reference.patient_id.isin(roles.patient_id)]\n    columns = [\'image_id\', \'patient_id\', \'laterality\', \'view\', \'decoded_pixel_sha256\']\n    require(manifest.image_id.is_unique and len(manifest) == complete[\'n_images\'] and\n            manifest[columns].sort_values(\'image_id\').reset_index(drop=True).equals(\n                expected[columns].sort_values(\'image_id\').reset_index(drop=True)),\n            \'Pilot image coverage or original pixels differ from the full audit.\')\n    paths = []\n    for relative in manifest.cache_relative_path:\n        path = Path(relative)\n        require(not path.is_absolute() and \'..\' not in path.parts, \'Invalid pilot cache path.\')\n        paths.append(str(root / path))\n        require(Path(paths[-1]).is_file(), \'A pilot cached image is missing.\')\n    manifest[\'cache_path\'] = paths\n    return manifest\n\n\ndef run_pilot(run_root: Path, breasts: pd.DataFrame, config: dict, trial: str,\n              model_overrides: dict | None = None, cache_root: Path | None = None) -> dict:\n    from .pipeline import require_pixel_audit\n    from .training import train_member\n    require(re.fullmatch(r\'[a-zA-Z0-9_-]+\', trial) is not None, \'Use a simple trial name.\')\n    require(not (run_root / \'frozen_protocol.json\').exists(), \'Global development is closed after the protocol freeze.\')\n    reference = require_pixel_audit(run_root, config)\n    roles = pd.read_csv(run_root / \'pilot\' / \'roles_private.csv\')\n    require((roles.cohort == \'pilot\').all() and (roles.outer_fold == -1).all(), \'Nonpilot patients entered development.\')\n    model_config = {**config[\'model\'], **(model_overrides or {})}\n    require(set(model_config) == set(config[\'model\']), \'Unknown pilot model option.\')\n    metadata_sha256 = json.loads((run_root / \'experiment.json\').read_text())[\'metadata_sha256\']\n    if cache_root is not None:\n        manifest = load_pilot_cache(cache_root, reference, roles, model_config, metadata_sha256)\n    else:\n        require((model_config[\'height\'], model_config[\'width\']) ==\n                (config[\'model\'][\'height\'], config[\'model\'][\'width\']), \'Attach a matching higher-resolution pilot cache.\')\n        manifest = reference.loc[reference.patient_id.isin(roles.patient_id)].copy()\n    subset = breasts.loc[breasts.patient_id.isin(roles.patient_id)]\n    folder = run_root / \'pilot\' / \'trials\' / trial\n    folder.mkdir(parents=True, exist_ok=True)\n    trial_record = {\'purpose\': \'development_only\', \'trial\': trial, \'model\': model_config,\n                    \'seed\': config[\'ensemble_seeds\'][0], \'metadata_sha256\': metadata_sha256,\n                    \'role_sha256\': hashlib.sha256(roles.to_csv(index=False).encode()).hexdigest(),\n                    \'final_test_results_opened\': False}\n    identity_path = folder / \'pilot_trial.json\'\n    if identity_path.exists():\n        require(json.loads(identity_path.read_text()) == trial_record, \'Use a new name for changed pilot settings.\')\n    else:\n        write_json(identity_path, trial_record)\n    roles.to_csv(folder / \'roles_private.csv\', index=False)\n    seed = config[\'ensemble_seeds\'][0]\n    result = train_member(manifest, subset, roles, model_config, seed, folder / f\'seed_{seed}\', metadata_sha256)\n    return {**result, \'purpose\': \'development_only\', \'trial\': trial,\n            \'interpretation\': \'Model development; these patients are excluded from final test endpoints.\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-root\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--config\', type=Path, required=True)\n    parser.add_argument(\'--height\', type=int, default=1536)\n    parser.add_argument(\'--width\', type=int, default=1152)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    args = parser.parse_args()\n    prepare_pilot_cache(args.data_root or locate_dataset(), args.output,\n                        json.loads(args.config.read_text()), args.height, args.width, args.workers)\n\n\nif __name__ == \'__main__\':\n    main()\n')


In [ ]:
# Audited source: pipeline.py
(PROJECT / 'mammo_reliability' / 'pipeline.py').write_text('"""Command-line workflow. Audits and training records stay in a private directory."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .audit import (BREAST_KEY, STANDARD_VIEWS, audit_metadata, environment, file_sha256,\n                    locate_dataset, require, write_json)\nfrom .evaluation import (MonotonePlatt, add_review_scores, classification_metrics, ensemble_features,\n                         paired_cluster_bootstrap, review_at_budget, review_selection, sensitivity_threshold)\nfrom .study import (freeze_protocol, make_study_patients, pilot_roles,\n                    require_frozen_protocol, study_outer_roles)\n\n\ndef initialize(data_root: Path, run_root: Path, config: dict):\n    images = pd.read_csv(data_root / "train.csv")\n    breasts, summary, tables = audit_metadata(images)\n    patients = make_study_patients(breasts, config["n_outer_folds"], config["split_seed"],\n                                   config["pilot_fraction"], config["pilot_tune_fraction"])\n    run_root.mkdir(parents=True, exist_ok=True)\n    identity = {"config": config, "metadata_sha256": file_sha256(data_root / "train.csv")}\n    identity_path = run_root / "experiment.json"\n    if identity_path.exists():\n        require(json.loads(identity_path.read_text()) == identity, "Different source/config already exists in this directory.")\n    else:\n        write_json(identity_path, identity)\n    (run_root / "aggregates").mkdir(exist_ok=True)\n    write_json(run_root / "aggregates" / "data_audit.json", summary)\n    write_json(run_root / "environment.json", environment())\n    for name, table in tables.items():\n        table.to_csv(run_root / "aggregates" / f"{name}.csv", index=False)\n    breasts.to_csv(run_root / "breasts_private.csv", index=False)\n    images.loc[images.view.isin(STANDARD_VIEWS), [\'patient_id\', \'image_id\', \'laterality\', \'view\']].sort_values(\n        \'image_id\').to_csv(run_root / \'standard_images_private.csv\', index=False)\n    patients.to_csv(run_root / "patients_private.csv", index=False)\n    pilot_folder = run_root / \'pilot\'\n    pilot_folder.mkdir(exist_ok=True)\n    pilot_roles(patients).to_csv(pilot_folder / \'roles_private.csv\', index=False)\n    patients.groupby([\'cohort\', \'site_id\', \'patient_cancer\']).size().rename(\'patients\').to_csv(\n        run_root / \'aggregates\' / \'cohort_patient_counts.csv\')\n    breasts.merge(patients[[\'patient_id\', \'cohort\', \'pilot_role\']], on=\'patient_id\', validate=\'many_to_one\').groupby(\n        [\'cohort\', \'pilot_role\', \'site_id\', \'cancer\']).size().rename(\'breasts\').to_csv(\n            run_root / \'aggregates\' / \'cohort_breast_counts.csv\')\n    for fold in range(config["n_outer_folds"]):\n        folder = run_root / f"fold_{fold}"\n        folder.mkdir(exist_ok=True)\n        roles = study_outer_roles(patients, fold, config["split_seed"])\n        roles.to_csv(folder / "roles_private.csv", index=False)\n        roles.groupby(["role", "site_id", "patient_cancer"]).size().rename("patients").to_csv(folder / "partition_counts.csv")\n    return images, breasts, patients, summary\n\n\ndef require_pixel_audit(run_root: Path, config: dict):\n    from .imaging import check_cross_patient_duplicates\n    root = run_root / "preprocessed"\n    complete = json.loads((root / "pixel_audit_complete.json").read_text())\n    settings = json.loads((root / "preprocessing.json").read_text())\n    identity = json.loads((run_root / "experiment.json").read_text())\n    require(complete["metadata_sha256"] == identity["metadata_sha256"], "Pixel audit uses another dataset.")\n    require(settings["height"] == config["model"]["height"] and settings["width"] == config["model"]["width"], "Preprocessing resolution differs from model.")\n    require(file_sha256(root / "manifest.csv") == complete["manifest_sha256"], "Pixel manifest changed after audit.")\n    manifest = pd.read_csv(root / "manifest.csv")\n    require(len(manifest) == complete["n_images"], "Pixel audit count mismatch.")\n    expected = pd.read_csv(run_root / \'standard_images_private.csv\')\n    columns = [\'patient_id\', \'image_id\', \'laterality\', \'view\']\n    require(manifest.image_id.is_unique and\n            manifest[columns].sort_values(\'image_id\').reset_index(drop=True).equals(expected[columns]),\n            \'Pixel audit does not cover the complete study image cohort.\')\n    check_cross_patient_duplicates(manifest)\n    return manifest\n\n\ndef fit_fold(run_root: Path, breasts: pd.DataFrame, config: dict, fold: int):\n    from .training import train_member\n    require_frozen_protocol(run_root, config)\n    manifest = require_pixel_audit(run_root, config)\n    folder = run_root / f"fold_{fold}"\n    roles = pd.read_csv(folder / "roles_private.csv")\n    metadata_hash = json.loads((run_root / "experiment.json").read_text())["metadata_sha256"]\n    for seed in config["ensemble_seeds"]:\n        train_member(manifest, breasts, roles, config["model"], seed, folder / f"seed_{seed}", metadata_hash)\n\n\ndef evaluate_fold(run_root: Path, breasts: pd.DataFrame, config: dict, fold: int):\n    from .training import predict_members, source_fingerprint\n    require_frozen_protocol(run_root, config)\n    manifest = require_pixel_audit(run_root, config)\n    folder = run_root / f"fold_{fold}"\n    roles = pd.read_csv(folder / "roles_private.csv")\n    label_table = breasts.merge(roles[["patient_id", "role"]], on="patient_id", validate="many_to_one")\n    frames = {}\n    for role in ["calibrate", "test"]:\n        subset = label_table.loc[label_table["role"] == role].copy()\n        members = predict_members(manifest, subset, config["model"], config["ensemble_seeds"], folder)\n        require(len(members) == len(subset) * len(config["ensemble_seeds"]), "Incomplete breast/member predictions.")\n        members.to_csv(folder / f"{role}_member_predictions_private.csv", index=False)\n        frames[role] = ensemble_features(members, config["ensemble_seeds"]).merge(subset, on=BREAST_KEY, validate="one_to_one")\n    cal, test = frames["calibrate"], frames["test"]\n    calibrator = MonotonePlatt().fit(cal["p_raw"], cal["cancer"])\n    for frame in [cal, test]:\n        frame["p_calibrated"] = calibrator.predict(frame["p_raw"])\n    threshold = sensitivity_threshold(cal["p_calibrated"], cal["cancer"], config["sensitivity_target"])\n    scored = add_review_scores(cal, test, threshold)\n    scored["outer_fold"] = fold\n    scored["threshold"] = threshold\n    methods = [name for name in scored if name.startswith("score_")]\n    review_rows = []\n    for budget in config["review_budgets"]:\n        for method in methods:\n            review_rows.append({"outer_fold": fold, **review_at_budget(scored, method, budget, config["split_seed"])})\n    metrics = {"outer_fold": fold, "source_sha256": source_fingerprint(),\n               "calibration": {"slope": calibrator.slope, "intercept": calibrator.intercept,\n                               "n_breasts": len(cal), "n_positive": int(cal["cancer"].sum())},\n               "target_calibration_sensitivity": config["sensitivity_target"],\n               "calibration_operating_point": classification_metrics(cal["cancer"], cal["p_calibrated"], threshold),\n               "held_out_test": classification_metrics(scored["cancer"], scored["p_calibrated"], threshold)}\n    pd.DataFrame(review_rows).to_csv(folder / "review_results.csv", index=False)\n    subgroup_rows = []\n    # Subgroups inherit the review selections from the full test fold.\n    # Their budget is NOT reset, which would describe a different policy.\n    for method in methods:\n        selection = review_selection(scored, method, config["primary_review_budget"], config["split_seed"])\n        grouped = scored.copy()\n        grouped["selected"] = selection\n        grouped["density_group"] = grouped["density_group"].fillna("Missing/conflicting")\n        for group_columns in [["site_id"], ["site_id", "density_group"], ["site_id", "has_extra_images"]]:\n            for group_name, sub in grouped.groupby(group_columns):\n                fn = int(((sub["cancer"] == 1) & ~sub["predicted_positive"]).sum())\n                flagged = int(((sub["cancer"] == 1) & sub["selected"]).sum())\n                subgroup_rows.append({"outer_fold": fold, "group_by": "+".join(group_columns),\n                                      "group": str(group_name), "method": method, "breasts": len(sub),\n                                      "positive_breasts": int(sub["cancer"].sum()), "false_negatives": fn,\n                                      "reviewed": int(sub["selected"].sum()), "false_negatives_flagged": flagged,\n                                      "fn_capture_fraction": flagged / fn if fn else None})\n    pd.DataFrame(subgroup_rows).to_csv(folder / "subgroup_review_results.csv", index=False)\n    scored.to_csv(folder / "test_scored_private.csv", index=False)\n    write_json(folder / "evaluation.json", metrics)\n    return metrics\n\n\ndef summarize(run_root: Path, breasts: pd.DataFrame, config: dict) -> dict:\n    require_frozen_protocol(run_root, config)\n    patients = pd.read_csv(run_root / \'patients_private.csv\')\n    breasts = breasts.loc[breasts.patient_id.isin(patients.loc[patients.cohort == \'evaluation\', \'patient_id\'])]\n    frames, reviews, checks = [], [], []\n    for fold in range(config["n_outer_folds"]):\n        folder = run_root / f"fold_{fold}"\n        frames.append(pd.read_csv(folder / "test_scored_private.csv"))\n        reviews.append(pd.read_csv(folder / "review_results.csv"))\n        checks.append(json.loads((folder / "evaluation.json").read_text()))\n    all_test = pd.concat(frames, ignore_index=True)\n    require(not all_test[BREAST_KEY].duplicated().any(), "A breast occurs in multiple held-out folds.")\n    require(set(map(tuple, all_test[BREAST_KEY].to_numpy())) == set(map(tuple, breasts[BREAST_KEY].to_numpy())), "Cross-validation omitted or added breasts.")\n    require(len({item["source_sha256"] for item in checks}) == 1, "Code changed between evaluated folds.")\n    comparison = paired_cluster_bootstrap(all_test, budget=config["primary_review_budget"],\n                                          repeats=config["bootstrap_repeats"], seed=config["split_seed"])\n    pd.concat(reviews, ignore_index=True).to_csv(run_root / "aggregates" / "all_fold_review_results.csv", index=False)\n    write_json(run_root / "aggregates" / "primary_comparison.json", comparison)\n    return comparison\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("stage", choices=["audit", "prepare", "pilot", "freeze", "train", "evaluate", "summarize"])\n    parser.add_argument("--data-root", type=Path)\n    parser.add_argument("--run-root", type=Path, default=Path("/kaggle/working/mammo_private_v0_2"))\n    parser.add_argument("--config", type=Path, default=Path("config.json"))\n    parser.add_argument("--fold", type=int, default=0)\n    parser.add_argument(\'--trial\', default=\'baseline_1024_v1\')\n    parser.add_argument(\'--pilot-model-config\', type=Path)\n    parser.add_argument(\'--pilot-cache-root\', type=Path)\n    parser.add_argument(\'--development-review\', type=Path)\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text())\n    data_root = args.data_root or locate_dataset()\n    images, breasts, _, summary = initialize(data_root, args.run_root, config)\n    require(0 <= args.fold < config["n_outer_folds"], "Invalid fold index.")\n    if args.stage == "audit":\n        print(json.dumps(summary, indent=2))\n    elif args.stage == "prepare":\n        from .imaging import prepare_images\n        prepare_images(images, data_root, args.run_root / "preprocessed", config["model"]["height"],\n                       config["model"]["width"], config["cache_images"])\n    elif args.stage == "train":\n        fit_fold(args.run_root, breasts, config, args.fold)\n    elif args.stage == \'pilot\':\n        from .pilot import run_pilot\n        overrides = json.loads(args.pilot_model_config.read_text()) if args.pilot_model_config else {}\n        print(json.dumps(run_pilot(args.run_root, breasts, config, args.trial, overrides, args.pilot_cache_root), indent=2))\n    elif args.stage == \'freeze\':\n        require(args.development_review is not None, \'Provide the concrete development review JSON.\')\n        print(json.dumps(freeze_protocol(args.run_root, config, args.development_review), indent=2))\n    elif args.stage == "evaluate":\n        print(json.dumps(evaluate_fold(args.run_root, breasts, config, args.fold), indent=2))\n    else:\n        print(json.dumps(summarize(args.run_root, breasts, config), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
# Audited source: study.py
(PROJECT / 'mammo_reliability' / 'study.py').write_text('"""Keep outcome-guided development patients outside every final test fold."""\nfrom __future__ import annotations\n\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom .audit import (BREAST_KEY, file_sha256, make_patient_folds,\n                    partition_outer_fold, require, write_json)\n\n\ndef make_study_patients(breasts: pd.DataFrame, n_splits: int = 5,\n                        seed: int = 20260909, pilot_fraction: float = 0.20,\n                        pilot_tune_fraction: float = 0.25) -> pd.DataFrame:\n    require(0 < pilot_fraction < 0.5 and 0 < pilot_tune_fraction < 0.5,\n            "Invalid development cohort fractions.")\n    require(not breasts[BREAST_KEY].duplicated().any(), "Expected unique breasts.")\n    require(breasts[\'has_both_views\'].all(), "Resolve missing standard views first.")\n    patients = breasts.groupby(\'patient_id\', as_index=False).agg(\n        site_id=(\'site_id\', \'first\'), patient_cancer=(\'cancer\', \'max\'))\n    strata = patients.site_id.astype(str) + \':\' + patients.patient_cancer.astype(str)\n    evaluation_idx, pilot_idx = train_test_split(\n        patients.index, test_size=pilot_fraction, stratify=strata, random_state=seed + 7000)\n    pilot = patients.loc[pilot_idx]\n    pilot_strata = pilot.site_id.astype(str) + \':\' + pilot.patient_cancer.astype(str)\n    pilot_fit_idx, pilot_tune_idx = train_test_split(\n        pilot.index, test_size=pilot_tune_fraction, stratify=pilot_strata, random_state=seed + 7001)\n    evaluation_breasts = breasts.loc[breasts.patient_id.isin(patients.loc[evaluation_idx, \'patient_id\'])]\n    evaluation_folds = make_patient_folds(evaluation_breasts, n_splits, seed)\n    fold_map = evaluation_folds.set_index(\'patient_id\').outer_fold\n    patients[\'cohort\'] = \'evaluation\'\n    patients.loc[pilot_idx, \'cohort\'] = \'pilot\'\n    patients[\'outer_fold\'] = patients.patient_id.map(fold_map).fillna(-1).astype(int)\n    patients[\'pilot_role\'] = \'unused\'\n    patients.loc[pilot_fit_idx, \'pilot_role\'] = \'fit\'\n    patients.loc[pilot_tune_idx, \'pilot_role\'] = \'tune\'\n    require((patients.loc[patients.cohort == \'pilot\', \'outer_fold\'] == -1).all(),\n            "Development patients received a test fold.")\n    require((patients.loc[patients.cohort == \'evaluation\', \'outer_fold\'] >= 0).all(),\n            "An evaluation patient lacks a fold.")\n    return patients.sort_values(\'patient_id\').reset_index(drop=True)\n\n\ndef pilot_roles(patients: pd.DataFrame) -> pd.DataFrame:\n    result = patients.loc[patients.cohort == \'pilot\'].copy()\n    require(len(result) > 0 and result.patient_id.is_unique, "Invalid pilot patients.")\n    require(set(result.pilot_role.unique()) == {\'fit\', \'tune\'}, "Pilot role mismatch.")\n    result[\'role\'] = result.pilot_role\n    return result.reset_index(drop=True)\n\n\ndef study_outer_roles(patients: pd.DataFrame, fold: int, seed: int = 20260909) -> pd.DataFrame:\n    require(patients.patient_id.is_unique, "Patient identifiers repeat.")\n    evaluation = patients.loc[patients.cohort == \'evaluation\'].reset_index(drop=True)\n    roles = partition_outer_fold(evaluation, fold, seed)\n    # After global settings are fixed, pilot patients contribute to final model\n    # fitting. They never contribute to tune, calibration, or test endpoints.\n    pilot = pilot_roles(patients)\n    pilot[\'role\'] = \'fit\'\n    result = pd.concat([roles, pilot], ignore_index=True).sort_values(\'patient_id\').reset_index(drop=True)\n    require(set(result.patient_id) == set(patients.patient_id), "Partition omitted patients.")\n    require((result.loc[result.cohort == \'pilot\', \'role\'] == \'fit\').all(),\n            "Pilot patients leaked into an evaluation role.")\n    return result\n\n\ndef partition_identity(run_root: Path, config: dict) -> dict:\n    names = [\'experiment.json\', \'patients_private.csv\', \'standard_images_private.csv\', \'pilot/roles_private.csv\']\n    names += [f\'fold_{fold}/roles_private.csv\' for fold in range(config[\'n_outer_folds\'])]\n    return {name: file_sha256(run_root / name) for name in names}\n\n\ndef require_frozen_protocol(run_root: Path, config: dict) -> dict:\n    from .training import source_fingerprint\n    path = run_root / \'frozen_protocol.json\'\n    require(path.is_file(), "Final training/evaluation requires a reviewed, frozen protocol; run development pilots first.")\n    frozen = json.loads(path.read_text())\n    require(frozen[\'config\'] == config, "Configuration changed after protocol freeze.")\n    require(frozen[\'source_sha256\'] == source_fingerprint(), "Source changed after protocol freeze.")\n    require(frozen[\'partition_files\'] == partition_identity(run_root, config),\n            "Patient assignments changed after protocol freeze.")\n    require(frozen[\'development_review_sha256\'] == file_sha256(run_root / \'development_review.json\'),\n            "Development review changed after protocol freeze.")\n    return frozen\n\n\ndef freeze_protocol(run_root: Path, config: dict, review_path: Path) -> dict:\n    """Record a concrete development review before final-fold training starts."""\n    from .training import source_fingerprint\n    require(not any(run_root.glob(\'fold_*/test*predictions*\')) and\n            not any(run_root.glob(\'fold_*/test_scored_private.csv\')),\n            "Held-out predictions already exist; a retrospective freeze is invalid.")\n    require(not any(run_root.glob(\'fold_*/seed_*/signature.json\')),\n            "Freeze the protocol before starting final-fold training.")\n    review = json.loads(review_path.read_text())\n    for field in [\'decision\', \'resolution_assessment\', \'literature_assessment\', \'limitations\']:\n        require(isinstance(review.get(field), str) and bool(review[field].strip()),\n                f"Development review needs {field}.")\n    require(isinstance(review.get(\'completed_pilot_directories\'), list) and\n            len(review[\'completed_pilot_directories\']) > 0, "Review must reference completed pilot runs.")\n    evidence = []\n    expected_patients = pd.read_csv(run_root / \'pilot\' / \'roles_private.csv\')\n    metadata_sha256 = json.loads((run_root / \'experiment.json\').read_text())[\'metadata_sha256\']\n    for directory in review[\'completed_pilot_directories\']:\n        directory = Path(directory)\n        details = json.loads((directory / \'pilot_trial.json\').read_text())\n        require(details[\'purpose\'] == \'development_only\', "Expected an isolated development trial.")\n        require(details[\'metadata_sha256\'] == metadata_sha256, \'Reviewed pilot uses different source data.\')\n        roles = pd.read_csv(directory / \'roles_private.csv\')\n        require(roles.sort_values(\'patient_id\').reset_index(drop=True).equals(\n            expected_patients.sort_values(\'patient_id\').reset_index(drop=True)),\n            "Reviewed pilot uses different patients or roles.")\n        completed = list(directory.glob(\'seed_*/completed.json\'))\n        require(len(completed) > 0, "A referenced pilot has not completed training.")\n        require(all(json.loads(p.read_text()).get(\'status\') == \'training_completed\' for p in completed),\n                \'A referenced pilot did not complete successfully.\')\n        evidence.append({\'trial\': str(directory), \'trial_sha256\': file_sha256(directory / \'pilot_trial.json\'),\n                         \'completed_files\': {str(p): file_sha256(p) for p in completed}})\n    destination = run_root / \'development_review.json\'\n    if destination.exists():\n        require(json.loads(destination.read_text()) == review, "A different development review already exists.")\n    else:\n        write_json(destination, review)\n    record = {\'frozen_at_utc\': datetime.now(timezone.utc).isoformat(), \'config\': config,\n              \'source_sha256\': source_fingerprint(), \'partition_files\': partition_identity(run_root, config),\n              \'development_review_sha256\': file_sha256(destination), \'pilot_evidence\': evidence}\n    target = run_root / \'frozen_protocol.json\'\n    require(not target.exists(), "A frozen protocol already exists; do not overwrite it.")\n    write_json(target, record)\n    return record\n')


In [ ]:
# Audited source: training.py
(PROJECT / 'mammo_reliability' / 'training.py').write_text('"""Train on fit patients; select checkpoints using tune patients only."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nfrom .audit import BREAST_KEY, environment, file_sha256, require, write_json\nfrom .model import BreastDataset, MammographyModel, collate_breasts, seed_everything\n\n\ndef source_fingerprint() -> str:\n    h = hashlib.sha256()\n    for path in sorted(Path(__file__).parent.glob("*.py")):\n        h.update(path.name.encode())\n        h.update(path.read_bytes())\n    return h.hexdigest()\n\n\ndef loader(dataset, config: dict, training: bool, seed: int = 0) -> DataLoader:\n    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=training,\n                      num_workers=config["num_workers"], collate_fn=collate_breasts,\n                      pin_memory=torch.cuda.is_available(), persistent_workers=False,\n                      generator=torch.Generator().manual_seed(seed))\n\n\n@torch.inference_mode()\ndef predict(model: MammographyModel, dataset: BreastDataset, config: dict, device: torch.device) -> pd.DataFrame:\n    model.eval()\n    records = []\n    for batch in loader(dataset, config, False):\n        images, slots = batch["images"].to(device), batch["slots"].to(device)\n        _, view_logits = model(images, slots, len(batch["labels"]))\n        values = view_logits.cpu().numpy()\n        require(np.isfinite(values).all(), "Model emitted nonfinite predictions.")\n        records.extend({"patient_id": p, "laterality": l, "z_cc": float(z[0]), "z_mlo": float(z[1])}\n                       for p, l, z in zip(batch["patient_id"], batch["laterality"], values))\n    return pd.DataFrame(records)\n\n\ndef save_checkpoint(path: Path, state: dict) -> None:\n    temporary = path.with_suffix(".tmp")\n    torch.save(state, temporary)\n    temporary.replace(path)\n\n\ndef train_member(manifest: pd.DataFrame, breasts: pd.DataFrame, roles: pd.DataFrame, config: dict,\n                 seed: int, output: str | Path, metadata_sha256: str) -> dict:\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    require(roles["patient_id"].is_unique, "Patients occur in more than one role.")\n    labeled = breasts.merge(roles[["patient_id", "role"]], on="patient_id", validate="many_to_one")\n    fit, tune = labeled.loc[labeled["role"] == "fit"], labeled.loc[labeled["role"] == "tune"]\n    require(len(fit) > 0 and len(tune) > 0, "Empty fitting or tuning partition.")\n    require(set(fit["patient_id"]).isdisjoint(set(tune["patient_id"])), "Patient leakage between fitting and tuning.")\n    require(set(fit["cancer"].unique()) == {0, 1} and set(tune["cancer"].unique()) == {0, 1}, "Both partitions need both classes.")\n    role_hash = hashlib.sha256(roles.sort_values("patient_id").to_csv(index=False).encode()).hexdigest()\n    signature = {"config": config, "seed": int(seed), "metadata_sha256": metadata_sha256,\n                 "roles_sha256": role_hash, "source_sha256": source_fingerprint(),\n                 "manifest_sha256": hashlib.sha256(manifest.sort_values("image_id").to_csv(index=False).encode()).hexdigest()}\n    signature_path = output / "signature.json"\n    if signature_path.exists():\n        require(json.loads(signature_path.read_text()) == signature, "Run configuration/code changed; use a new run directory.")\n    else:\n        write_json(signature_path, signature)\n    if (output / "completed.json").exists():\n        finished = json.loads((output / "completed.json").read_text())\n        require(file_sha256(output / "best.pt") == finished["checkpoint_sha256"], "Completed model checkpoint changed.")\n        return finished\n\n    seed_everything(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if config.get("require_gpu", True):\n        require(device.type == "cuda", "GPU unavailable. CPU is supported for software tests; enable a Kaggle GPU for research training.")\n    model = MammographyModel(pretrained=config["pretrained"]).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)\n    amp_enabled = bool(config.get("mixed_precision", True) and device.type == "cuda")\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    n_pos = int(fit["cancer"].sum())\n    positive_weight = (len(fit) - n_pos) / n_pos\n    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=device))\n    train_data = BreastDataset(manifest, fit, True, config["height"], config["width"])\n    tune_data = BreastDataset(manifest, tune, False, config["height"], config["width"])\n    start_epoch, best_ap, stale, history = 0, -1.0, 0, []\n    last = output / "last.pt"\n    if last.exists():\n        # Only load checkpoints created by this run in a trusted private directory.\n        state = torch.load(last, map_location=device, weights_only=False)\n        require(state["signature"] == signature, "Resume checkpoint signature mismatch.")\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        scheduler.load_state_dict(state["scheduler"])\n        scaler.load_state_dict(state["scaler"])\n        start_epoch, best_ap, stale, history = state["next_epoch"], state["best_ap"], state["stale"], state["history"]\n\n    require(config["batch_size"] >= 1 and config["accumulation_steps"] >= 1, "Invalid batch settings.")\n    for epoch in range(start_epoch, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        epoch_seed = seed + 1000 * epoch\n        seed_everything(epoch_seed)\n        model.train()\n        batches = loader(train_data, config, True, epoch_seed)\n        optimizer.zero_grad(set_to_none=True)\n        total_loss, n_seen, start_time = 0.0, 0, time.monotonic()\n        if device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats(device)\n        accumulation = config["accumulation_steps"]\n        for i, batch in enumerate(batches):\n            images = batch["images"].to(device, non_blocking=True)\n            slots, labels = batch["slots"].to(device), batch["labels"].to(device)\n            group_start = (i // accumulation) * accumulation * config["batch_size"]\n            group_breasts = min(accumulation * config["batch_size"], len(train_data) - group_start)\n            with torch.autocast(device_type=device.type, enabled=amp_enabled):\n                logits, _ = model(images, slots, len(labels))\n                loss = loss_fn(logits, labels)\n            require(torch.isfinite(loss).item(), "Nonfinite training loss.")\n            scaler.scale(loss * len(labels) / group_breasts).backward()\n            if (i + 1) % accumulation == 0 or (i + 1) == len(batches):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n            total_loss += float(loss.detach()) * len(labels)\n            n_seen += len(labels)\n            if (i + 1) % 256 == 0:\n                print(json.dumps({\'epoch\': epoch + 1, \'fit_breasts_processed\': n_seen,\n                                  \'fit_breasts_total\': len(train_data)}), flush=True)\n        training_seconds = time.monotonic() - start_time\n        validation_start = time.monotonic()\n        predictions = predict(model, tune_data, config, device).merge(tune[BREAST_KEY + ["cancer"]], on=BREAST_KEY, validate="one_to_one")\n        logits = (predictions["z_cc"].to_numpy() + predictions["z_mlo"].to_numpy()) / 2\n        from scipy.special import expit\n        probabilities = expit(logits)\n        ap = float(average_precision_score(predictions["cancer"], probabilities))\n        record = {"epoch": epoch + 1, "weighted_training_loss": total_loss / n_seen,\n                  "tune_average_precision": ap, "tune_roc_auc": float(roc_auc_score(predictions["cancer"], probabilities)),\n                  "seconds": time.monotonic() - start_time, "learning_rate": optimizer.param_groups[0]["lr"]}\n        record.update(training_seconds=training_seconds, validation_seconds=time.monotonic() - validation_start,\n                      fit_breasts_per_second=n_seen / training_seconds,\n                      peak_gpu_allocated_gib=torch.cuda.max_memory_allocated(device) / 1024**3 if device.type == \'cuda\' else None)\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        if ap > best_ap:\n            best_ap, stale = ap, 0\n            save_checkpoint(output / "best.pt", {"state_dict": model.state_dict(), "epoch": epoch + 1, "seed": seed})\n        else:\n            stale += 1\n        scheduler.step(ap)\n        pd.DataFrame(history).to_csv(output / "learning_curve.csv", index=False)\n        save_checkpoint(last, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                              "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),\n                              "signature": signature, "next_epoch": epoch + 1, "best_ap": best_ap,\n                              "stale": stale, "history": history})\n    require((output / "best.pt").exists(), "No model checkpoint was produced.")\n    result = {"status": "training_completed", "best_tune_average_precision": best_ap,\n              "checkpoint_sha256": file_sha256(output / "best.pt"), "positive_class_weight": positive_weight,\n              "fit_breasts": len(fit), "tune_breasts": len(tune), "device": str(device), "environment": environment()}\n    write_json(output / "completed.json", result)\n    return result\n\n\ndef predict_members(manifest: pd.DataFrame, breasts: pd.DataFrame, config: dict, seeds: list[int],\n                    fold_dir: str | Path) -> pd.DataFrame:\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    dataset = BreastDataset(manifest, breasts, False, config["height"], config["width"])\n    frames = []\n    for seed in seeds:\n        folder = Path(fold_dir) / f"seed_{seed}"\n        completed = json.loads((folder / "completed.json").read_text())\n        require(file_sha256(folder / "best.pt") == completed["checkpoint_sha256"], "Prediction checkpoint changed.")\n        signature = json.loads((folder / "signature.json").read_text())\n        require(signature["source_sha256"] == source_fingerprint() and signature["config"] == config,\n                "Prediction code/config differs from training.")\n        require(signature["manifest_sha256"] == hashlib.sha256(manifest.sort_values("image_id").to_csv(index=False).encode()).hexdigest(),\n                "The image manifest differs from the one used for training.")\n        model = MammographyModel(pretrained=False).to(device)\n        model.load_state_dict(torch.load(folder / "best.pt", map_location=device, weights_only=True)["state_dict"])\n        member = predict(model, dataset, config, device)\n        member["seed"] = seed\n        frames.append(member)\n        del model\n    return pd.concat(frames, ignore_index=True)\n')


In [ ]:
CONFIG = {'experiment_id': 'rsna_reliability_v0_2', 'split_seed': 20260909, 'n_outer_folds': 5, 'pilot_fraction': 0.2, 'pilot_tune_fraction': 0.25, 'ensemble_seeds': [17, 41, 73], 'sensitivity_target': 0.9, 'review_budgets': [0.01, 0.05, 0.1, 0.2], 'primary_review_budget': 0.05, 'bootstrap_repeats': 2000, 'cache_images': True, 'model': {'height': 1024, 'width': 768, 'batch_size': 1, 'accumulation_steps': 8, 'num_workers': 2, 'epochs': 8, 'patience': 3, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'pretrained': True, 'mixed_precision': True, 'require_gpu': True}}
(PROJECT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<3',
            'scipy': 'scipy>=1.11,<2', 'sklearn': 'scikit-learn>=1.4,<2',
            'PIL': 'Pillow>=10,<13', 'pydicom': 'pydicom>=3,<4'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)


In [ ]:
dependencies = {'imageio':'imageio>=2.34,<3', 'h5py':'h5py>=3.10,<4',
                'cv2':'opencv-python-headless>=4.9,<5', 'matplotlib':'matplotlib>=3.8,<4',
                'pylibjpeg':'pylibjpeg>=2,<3', 'openjpeg':'pylibjpeg-openjpeg>=2,<3',
                'libjpeg':'pylibjpeg-libjpeg>=2,<3'}
missing = [spec for module,spec in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing], check=True)


In [ ]:
(PROJECT / 'gmic_adapter.py').write_text('"""Engineering adapter for a pinned, unmodified NYU GMIC implementation.\n\nThe upstream implementation and weights retain their authorship and AGPLv3\nlicense. This adapter is development code; it is not a clinical pipeline.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\nimport urllib.request\n\nimport numpy as np\n\nUPSTREAM_COMMIT = \'594029efe6184bcca3a151fa1dd46c5fd620cb25\'\nHISTORICAL_FIXTURE_COMMIT = \'febe2431dbe6efd75bcb37ddea85dc48c0df0b07\'\n\n\ndef filter_legacy_state(state, model_keys):\n    """Permit one unused saved tensor, never arbitrary missing/extra weights.\n\n    The pinned source has no shared_rep_filter module or forward-path use.\n    Its official loader uses strict=False. Our narrower exception preserves\n    strict loading for all live model parameters.\n    """\n    import copy\n    extra = set(state) - set(model_keys)\n    if extra - {\'shared_rep_filter.weight\'}:\n        raise ValueError(f\'Unexplained checkpoint tensors: {sorted(extra)}\')\n    missing = set(model_keys) - set(state)\n    if any(not name.endswith(\'.num_batches_tracked\') for name in missing):\n        raise ValueError(f\'Missing active checkpoint tensors: {sorted(missing)}\')\n    filtered = copy.copy(state)\n    for name in extra:\n        del filtered[name]\n    return filtered, sorted(extra)\n\n\ndef digest_file(path, algorithm=\'sha256\', git_blob=False):\n    path = Path(path)\n    digest = hashlib.new(algorithm)\n    if git_blob:\n        digest.update(b\'blob \' + str(path.stat().st_size).encode() + b\'\\0\')\n    with path.open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef acquire_upstream(manifest_path, destination):\n    """Check every file against the blob identity in the inspected Git tree."""\n    manifest = json.loads(Path(manifest_path).read_text())\n    if manifest[\'commit\'] not in {UPSTREAM_COMMIT, HISTORICAL_FIXTURE_COMMIT}:\n        raise ValueError(\'Unexpected GMIC revision.\')\n    destination = Path(destination)\n    records = []\n    for relative, blob in manifest[\'files\'].items():\n        relative_path = Path(relative)\n        if relative_path.is_absolute() or \'..\' in relative_path.parts:\n            raise ValueError(\'Invalid upstream path.\')\n        path = destination / relative_path\n        path.parent.mkdir(parents=True, exist_ok=True)\n        if not path.exists():\n            url = f"https://raw.githubusercontent.com/nyukat/GMIC/{manifest[\'commit\']}/{relative}"\n            temporary = path.with_suffix(path.suffix + \'.download\')\n            with urllib.request.urlopen(url, timeout=120) as source, temporary.open(\'wb\') as target:\n                shutil.copyfileobj(source, target, length=1024*1024)\n            if digest_file(temporary, \'sha1\', git_blob=True) != blob:\n                raise ValueError(f\'Upstream content mismatch: {relative}\')\n            temporary.replace(path)\n        if digest_file(path, \'sha1\', git_blob=True) != blob:\n            raise ValueError(f\'Existing upstream content mismatch: {relative}\')\n        records.append({\'path\': relative, \'git_blob\': blob, \'bytes\': path.stat().st_size,\n                        \'sha256\': digest_file(path)})\n        print(\'GMIC VERIFIED FILE\', relative, path.stat().st_size, flush=True)\n    return records\n\n\ndef infer_horizontal_flip(image, view):\n    """Provisional, label-free orientation heuristic requiring visual review.\n\nCompare intensity mass in outer fifths of the original image. The brighter\nside is the proposed chest-wall side. This is NOT a validated orientation\ndetector. Ambiguity is recorded, not used to exclude patients silently.\n"""\n    image = np.asarray(image)\n    if image.ndim != 2 or min(image.shape) < 5 or view not in (\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\'):\n        raise ValueError(\'Invalid image or standard view.\')\n    if not np.isfinite(image).all() or image.min() < 0 or image.max() <= image.min():\n        raise ValueError(\'Invalid image intensity range.\')\n    width = max(1, image.shape[1] // 5)\n    left = float(image[:, :width].sum(dtype=np.float64))\n    right = float(image[:, -width:].sum(dtype=np.float64))\n    total = left + right\n    if total <= 0:\n        raise ValueError(\'No edge signal for orientation.\')\n    wall_left = left >= right\n    usual_wall_left = view.startswith(\'L-\')\n    return (\'NO\' if wall_left == usual_wall_left else \'YES\'), {\n        \'proposed_chest_wall_side\': \'left\' if wall_left else \'right\',\n        \'edge_mass_contrast\': abs(left-right)/total,\n        \'left_edge_mass\': left, \'right_edge_mass\': right,\n        \'orientation_requires_visual_review\': True,\n    }\n\n\ndef preprocess_array(image, view, horizontal_flip):\n    """Use original crop, center, orientation, padding and normalization code."""\n    from src.cropping.crop_mammogram import crop_img_from_largest_connected, image_orientation\n    from src.optimal_centers.get_optimal_centers import extract_center\n    from src.data_loading import loading, augmentations\n\n    info = crop_img_from_largest_connected(\n        image, image_orientation(horizontal_flip, view[0]), True, 100, 50, 1/3)\n    top, bottom, left, right = map(int, info[0])\n    crop = image[top:bottom, left:right]\n    metadata = {\'full_view\': view, \'view\': view[2:], \'side\': view[0],\n                \'horizontal_flip\': horizontal_flip, \'window_location\': info[0],\n                \'rightmost_points\': info[1], \'bottommost_points\': info[2],\n                \'distance_from_starting_side\': info[3]}\n    center = extract_center(metadata, crop)\n    oriented = loading.flip_image(crop.astype(np.float32), view, horizontal_flip)\n    window, _ = augmentations.random_augmentation_best_center(\n        oriented, (2944, 1920), np.random.RandomState(0), best_center=center, view=view)\n    # Independently check the helper against the exact upstream inference entry.\n    processed = loading.process_image(oriented, view, center)\n    expected = window.copy()\n    loading.standard_normalize_single_image(expected)\n    if not np.array_equal(processed, expected):\n        raise ValueError(\'Adapter changed upstream image processing.\')\n    if processed.shape != (2944, 1920) or not np.isfinite(processed).all() or processed.std() < .99:\n        raise ValueError(\'Invalid processed GMIC input.\')\n    nonzero = max(int(np.count_nonzero(image)), 1)\n    details = {\'raw_shape\': list(image.shape), \'crop_shape\': list(crop.shape),\n               \'window_location\': [top, bottom, left, right],\n               \'best_center\': [int(x) for x in center],\n               \'distance_from_starting_side\': int(info[3]),\n               \'nonzero_fraction_retained_after_crop\': float(np.count_nonzero(crop)/nonzero),\n               \'nonzero_fraction_retained_in_window\': float(np.count_nonzero(window)/nonzero),\n               \'horizontal_flip\': horizontal_flip,\n               \'normalized_mean\': float(processed.mean()), \'normalized_std\': float(processed.std()),\n               \'processed_sha256\': hashlib.sha256(processed.tobytes()).hexdigest()}\n    return processed, window, details\n\n\ndef decode_native(path):\n    """Keep native pixel dimensions; no reuse/upscaling of the small cache."""\n    import pydicom\n    from pydicom.pixels import apply_modality_lut, apply_voi_lut\n    from mammo_reliability.imaging import normalize_pixels\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim == 3 and raw.shape[0] == 1:\n        raw = raw[0]\n    if raw.ndim != 2 or \'PresentationLUTSequence\' in ds:\n        raise ValueError(\'Unsupported DICOM presentation.\')\n    normalized = normalize_pixels(raw, apply_voi_lut(apply_modality_lut(raw, ds), ds),\n                                  str(ds.PhotometricInterpretation), ds.get(\'PixelPaddingValue\'),\n                                  ds.get(\'PixelPaddingRangeLimit\'))\n    image = np.round(normalized*65535).astype(np.uint16)\n    details = {\'photometric\': str(ds.PhotometricInterpretation),\n               \'transfer_syntax\': str(ds.file_meta.TransferSyntaxUID),\n               \'patient_orientation\': list(ds.get(\'PatientOrientation\', [])),\n               \'pixel_spacing\': [float(x) for x in ds.get(\'PixelSpacing\', [])],\n               \'imager_pixel_spacing\': [float(x) for x in ds.get(\'ImagerPixelSpacing\', [])],\n               \'manufacturer\': str(ds.get(\'Manufacturer\', \'\')),\n               \'voi_present\': bool(\'VOILUTSequence\' in ds or \'WindowCenter\' in ds)}\n    return image, details\n')
(PROJECT / 'gmic_preflight.py').write_text('"""Reference reproduction and fit-only engineering check, no RSNA accuracy claim."""\nfrom pathlib import Path\nimport json\nimport os\nimport pickle\nimport re\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\n\nfrom gmic_adapter import (UPSTREAM_COMMIT, HISTORICAL_FIXTURE_COMMIT, acquire_upstream, digest_file, decode_native,\n                          infer_horizontal_flip, preprocess_array, filter_legacy_state)\nfrom mammo_reliability.audit import environment, locate_dataset, require, write_json\nfrom mammo_reliability.pipeline import initialize\nfrom mammo_reliability.training import source_fingerprint\n\nROOT = Path(\'/kaggle/working/gmic_transfer_preflight\')\nSETTINGS = {\n    \'purpose\': \'reference_reproduction_and_fit_only_technical_preflight\',\n    \'upstream_commit\': UPSTREAM_COMMIT,\n    \'models\': [1, 2, 3, 4, 5], \'weights_frozen\': True, \'precision\': \'FP32\',\n    \'upstream_source_modifications\': [],\n    \'checkpoint_compatibility\': \'Discard only unused shared_rep_filter.weight if present; strictly load active tensors.\',\n    \'reference\': \'README table with default run.sh MODEL_INDEX=1\',\n    \'reference_max_absolute_error_gate\': 0.005,\n    \'reference_gate_limit\': \'Engineering tolerance; not an accuracy test or exact old-environment replication.\',\n    \'rsna_sample\': \'Pilot fit only; 2 images per site/view/laterality plus 1 per machine, deduplicated.\',\n    \'sample_seed\': 20260924,\n    \'orientation\': \'provisional label-free outer-fifth intensity-mass heuristic; visual review required\',\n    \'rsna_preprocessing\': \'native DICOM presentation, 16-bit conversion, unmodified NYU crop/center/normalize\',\n    \'upstream_crop_erosion_iterations\': 100, \'upstream_crop_buffer_pixels\': 50,\n    \'image_size\': [2944, 1920], \'patches\': 6, \'patch_size\': [256, 256],\n    \'test_time_augmentation\': \'one deterministic center, matching released inference code\',\n    \'paper_replication\': False,\n    \'rsna_tuning_predictions\': False, \'final_test_results_opened\': False,\n    \'automatic_full_cohort_run\': False,\n    \'on_reference_mismatch\': \'Retain current-fixture failure; require historical reference agreement and current CLI parity.\',\n    \'gate_revision\': \'Historical fixture must match README; current adapter must match original current CLI inputs/outputs.\',\n    \'historical_fixture_commit\': HISTORICAL_FIXTURE_COMMIT,\n    \'current_cli_max_rounding_difference\': 0.00015,\n}\n\n\ndef load_models(upstream):\n    from src.modeling.gmic import GMIC\n    from src.constants import PERCENT_T_DICT\n    models = []\n    loading_records = []\n    for index in SETTINGS[\'models\']:\n        params = {\'device_type\': \'gpu\', \'gpu_number\': 0, \'cam_size\': (46, 30),\n                  \'K\': 6, \'crop_shape\': (256, 256), \'post_processing_dim\': 256,\n                  \'num_classes\': 2, \'use_v1_global\': False,\n                  \'percent_t\': PERCENT_T_DICT[str(index)]}\n        model = GMIC(params)\n        state = torch.load(upstream/\'models\'/f\'sample_model_{index}.p\', map_location=\'cpu\', weights_only=True)\n        active_state, ignored = filter_legacy_state(state, model.state_dict().keys())\n        model.load_state_dict(active_state, strict=True)\n        loading_records.append({\'model\': index, \'ignored_unused_keys\': ignored,\n                                \'ignored_tensor_shapes\': {k:list(state[k].shape) for k in ignored},\n                                \'active_state_loaded_strictly\': True})\n        model.eval().to(\'cuda\')\n        for parameter in model.parameters():\n            parameter.requires_grad_(False)\n        models.append(model)\n        print(\'GMIC MODEL LOADED\', index, flush=True)\n    write_json(ROOT/\'checkpoint_loading.json\', loading_records)\n    print(\'GMIC CHECKPOINT LOADING\', json.dumps(loading_records), flush=True)\n    return models\n\n\ndef infer(models, array):\n    x = torch.from_numpy(np.ascontiguousarray(array))[None, None].to(\'cuda\')\n    predictions = []\n    with torch.inference_mode():\n        for model in models:\n            out = model(x)\n            require(out.shape == (1, 2) and torch.isfinite(out).all().item(), \'Invalid model output.\')\n            require(torch.isfinite(model.saliency_map).all().item(), \'Invalid saliency map.\')\n            require(np.isfinite(model.patch_locations).all(), \'Invalid crop coordinates.\')\n            require(((out >= 0) & (out <= 1)).all().item(), \'Output outside [0,1].\')\n            predictions.append(out.cpu().numpy()[0].tolist())\n    return np.asarray(predictions)\n\n\ndef reproduce_reference(upstream, models):\n    expected = {}\n    for line in (upstream/\'README.md\').read_text().splitlines():\n        if re.match(r\'^[0-3]_[LR]-(CC|MLO)\\s*\\|\', line):\n            fields = [x.strip() for x in line.split(\'|\')]\n            expected[fields[0]] = [float(fields[1]), float(fields[2])]\n    require(len(expected) == 16, \'Reference table changed.\')\n    with (upstream/\'sample_data/exam_list_before_cropping.pkl\').open(\'rb\') as stream:\n        exams = pickle.load(stream)  # Pinned, blob-verified public author fixture.\n    records = []\n    for exam in exams:\n        for view in [\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\']:\n            name = exam[view][0]\n            raw = np.asarray(Image.open(upstream/\'sample_data/images\'/f\'{name}.png\'))\n            processed, _, details = preprocess_array(raw, view, exam[\'horizontal_flip\'])\n            predictions = infer(models, processed)\n            records.append({\'image\': name, \'expected_default_model\': expected[name],\n                            \'predictions_by_model\': predictions.tolist(),\n                            \'model_1_max_absolute_error\': float(np.max(np.abs(predictions[0]-expected[name]))),\n                            \'preprocessing\': details})\n            print(\'GMIC REFERENCE IMAGE\', name, records[-1][\'model_1_max_absolute_error\'], flush=True)\n    require({r[\'image\'] for r in records} == set(expected), \'Reference images missing.\')\n    max_error = max(r[\'model_1_max_absolute_error\'] for r in records)\n    result = {\'images\': len(records), \'max_absolute_error\': max_error,\n              \'tolerance\': SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'passed\': max_error <= SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'records\': records}\n    write_json(ROOT/\'reference_reproduction.json\', result)\n    print(\'GMIC REFERENCE RESULT\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    if not result[\'passed\']:\n        parity = diagnose_original_pipeline(upstream, result)\n        historical = reproduce_historical_reference(models, expected)\n        require(parity[\'all_preprocessed_arrays_identical\'] and\n                parity[\'max_adapter_vs_cli_error\'] <= SETTINGS[\'current_cli_max_rounding_difference\'],\n                \'Adapter does not reproduce the original current pipeline.\')\n        require(historical[\'passed\'], \'Historical fixtures do not reproduce the published reference.\')\n        result[\'compatibility_passed\'] = True\n        result[\'historical_reference_max_error\'] = historical[\'max_absolute_error\']\n        result[\'current_cli_max_error\'] = parity[\'max_adapter_vs_cli_error\']\n        result[\'interpretation\'] = \'Current fixture/table mismatch retained; original fixtures and current adapter parity verified.\'\n    else:\n        result[\'compatibility_passed\'] = True\n    write_json(ROOT/\'reference_reproduction.json\', result)\n    return result\n\n\ndef diagnose_original_pipeline(upstream, reference):\n    """Compare against the author\'s complete released entry points, unchanged."""\n    destination = ROOT/\'official_cli_diagnostic\'\n    cropped = destination/\'cropped\'\n    metadata = destination/\'data.pkl\'\n    predictions = destination/\'inference\'\n    destination.mkdir(exist_ok=True)\n    env = os.environ.copy()\n    env[\'PYTHONPATH\'] = str(upstream)\n    commands = [\n        [\'src/cropping/crop_mammogram.py\', \'--input-data-folder\', str(upstream/\'sample_data/images\'),\n         \'--output-data-folder\', str(cropped), \'--exam-list-path\', str(upstream/\'sample_data/exam_list_before_cropping.pkl\'),\n         \'--cropped-exam-list-path\', str(destination/\'cropped.pkl\'), \'--num-processes\', \'2\'],\n        [\'src/optimal_centers/get_optimal_centers.py\', \'--cropped-exam-list-path\', str(destination/\'cropped.pkl\'),\n         \'--data-prefix\', str(cropped), \'--output-exam-list-path\', str(metadata), \'--num-processes\', \'2\'],\n        [\'src/scripts/run_model.py\', \'--model-path\', str(upstream/\'models\'), \'--data-path\', str(metadata),\n         \'--image-path\', str(cropped), \'--segmentation-path\', str(destination/\'no_segmentations\'),\n         \'--output-path\', str(predictions), \'--device-type\', \'gpu\', \'--gpu-number\', \'0\', \'--model-index\', \'1\'],\n    ]\n    print(\'GMIC ORIGINAL CLI DIAGNOSTIC STARTING\', flush=True)\n    for command in commands:\n        subprocess.run([sys.executable, \'-u\', *command], cwd=upstream, env=env, check=True)\n    from src.data_loading import loading\n    original = pd.read_csv(predictions/\'predictions.csv\').set_index(\'image_index\')\n    with metadata.open(\'rb\') as stream: exams = pickle.load(stream)\n    original_inputs = {}\n    for exam in exams:\n        for view in [\'L-CC\', \'R-CC\', \'L-MLO\', \'R-MLO\']:\n            name = exam[view][0]\n            image = loading.load_image(str(cropped/f\'{name}.png\'), view, exam[\'horizontal_flip\'])\n            array = loading.process_image(image, view, exam[\'best_center\'][view][0])\n            import hashlib\n            original_inputs[name] = {\'processed_sha256\': hashlib.sha256(array.tobytes()).hexdigest(),\n                                     \'best_center\': [int(x) for x in exam[\'best_center\'][view][0]]}\n    rows = []\n    for record in reference[\'records\']:\n        name = record[\'image\']\n        scores = original.loc[name, [\'benign_pred\', \'malignant_pred\']].to_numpy(float)\n        row = {\'image\': name, \'original_cli_predictions\': scores.tolist(),\n               \'adapter_predictions\': record[\'predictions_by_model\'][0],\n               \'published_predictions\': record[\'expected_default_model\'],\n               \'adapter_vs_cli_max_error\': float(np.max(np.abs(scores-record[\'predictions_by_model\'][0]))),\n               \'cli_vs_published_max_error\': float(np.max(np.abs(scores-record[\'expected_default_model\']))),\n               \'preprocessed_arrays_identical\': original_inputs[name][\'processed_sha256\'] == record[\'preprocessing\'][\'processed_sha256\'],\n               \'original_center\': original_inputs[name][\'best_center\'],\n               \'adapter_center\': record[\'preprocessing\'][\'best_center\']}\n        rows.append(row)\n        print(\'GMIC ORIGINAL CLI COMPARISON\', json.dumps(row), flush=True)\n    result = {\'status\': \'diagnosis_only_original_reference_gate_unchanged\', \'records\': rows,\n              \'all_preprocessed_arrays_identical\': all(x[\'preprocessed_arrays_identical\'] for x in rows),\n              \'max_adapter_vs_cli_error\': max(x[\'adapter_vs_cli_max_error\'] for x in rows),\n              \'max_cli_vs_published_error\': max(x[\'cli_vs_published_max_error\'] for x in rows),\n              \'rsna_inference_allowed\': False}\n    write_json(ROOT/\'original_cli_comparison.json\', result)\n    print(\'GMIC ORIGINAL CLI SUMMARY\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    return result\n\n\ndef reproduce_historical_reference(models, current_expected):\n    """Use exact already-windowed author fixtures for the original README table.\n\n    Git history shows their replacement in December 2020. The initial author\n    inference code only converted these arrays to float and standardized them;\n    it explicitly disabled flipping. The checkpoint blobs are unchanged.\n    """\n    from src.data_loading.loading import standard_normalize_single_image\n    destination = ROOT/\'historical_fixtures\'\n    files = acquire_upstream(\'gmic_historical_manifest.json\', destination)\n    write_json(ROOT/\'verified_historical_files.json\', files)\n    expected = {}\n    for line in (destination/\'README.md\').read_text().splitlines():\n        if re.match(r\'^[0-3]_[LR]-(CC|MLO)\\s*\\|\', line):\n            parts = [x.strip() for x in line.split(\'|\')]\n            expected[parts[0]] = [float(parts[1]), float(parts[2])]\n    require(expected == current_expected, \'The historical and current reference tables differ.\')\n    records = []\n    for name in expected:\n        array = np.array(Image.open(destination/\'sample_data/cropped_images\'/f\'{name}.png\'), dtype=np.float32)\n        require(array.shape == (2944, 1920), \'Historical author fixture has unexpected dimensions.\')\n        standard_normalize_single_image(array)\n        predicted = infer([models[0]], array)[0]\n        record = {\'image\': name, \'expected\': expected[name], \'prediction\': predicted.tolist(),\n                  \'max_absolute_error\': float(np.max(np.abs(predicted-expected[name])))}\n        records.append(record)\n        print(\'GMIC HISTORICAL REFERENCE IMAGE\', json.dumps(record), flush=True)\n    max_error = max(x[\'max_absolute_error\'] for x in records)\n    result = {\'images\': len(records), \'fixture_commit\': HISTORICAL_FIXTURE_COMMIT,\n              \'max_absolute_error\': max_error, \'tolerance\': SETTINGS[\'reference_max_absolute_error_gate\'],\n              \'passed\': max_error <= SETTINGS[\'reference_max_absolute_error_gate\'], \'records\': records}\n    write_json(ROOT/\'historical_reference_reproduction.json\', result)\n    print(\'GMIC HISTORICAL REFERENCE SUMMARY\', json.dumps({k:v for k,v in result.items() if k != \'records\'}), flush=True)\n    return result\n\n\ndef check_rsna(models):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    data_root = locate_dataset()\n    config = json.loads(Path(\'config.json\').read_text())\n    metadata_root = ROOT/\'cohort_identity\'\n    initialize(data_root, metadata_root, config)\n    roles_path = metadata_root/\'pilot/roles_private.csv\'\n    require(digest_file(roles_path) == \'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\',\n            \'Reserved development roles changed.\')\n    roles = pd.read_csv(roles_path)\n    require(roles.patient_id.is_unique and set(roles.cohort) == {\'pilot\'} and (roles.outer_fold == -1).all(),\n            \'Invalid reserved pilot roles.\')\n    fit_ids = set(roles.loc[roles.role.eq(\'fit\'), \'patient_id\'])\n    images = pd.read_csv(data_root/\'train.csv\')\n    fit = images.loc[images.patient_id.isin(fit_ids) & images.view.isin([\'CC\', \'MLO\'])]\n    ordered = fit.sample(frac=1, random_state=SETTINGS[\'sample_seed\'])\n    sample = pd.concat([ordered.groupby([\'site_id\', \'view\', \'laterality\'], sort=True).head(2),\n                        ordered.groupby(\'machine_id\', sort=True).head(1)]).drop_duplicates(\'image_id\')\n    require(set(sample.patient_id).issubset(fit_ids), \'Engineering sample includes nonfit patients.\')\n    sample.to_csv(ROOT/\'engineering_sample_private.csv\', index=False)\n    records, thumbnails = [], []\n    for count, row in enumerate(sample.itertuples(index=False), 1):\n        started = time.monotonic()\n        raw, header = decode_native(data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\')\n        view = f\'{row.laterality}-{row.view}\'\n        flip, orientation = infer_horizontal_flip(raw, view)\n        processed, window, details = preprocess_array(raw, view, flip)\n        predictions = infer(models, processed)\n        record = {\'sample\': count, \'site_id\': int(row.site_id), \'machine_id\': int(row.machine_id),\n                  \'view\': view, **header, **orientation, **details,\n                  \'elapsed_seconds\': time.monotonic()-started}\n        records.append(record)\n        # No RSNA scores or cancer labels in aggregate logs or contact-sheet titles.\n        np.save(ROOT/f\'sample_{count:02d}_predictions_private.npy\', predictions)\n        thumbnails.append((np.asarray(Image.fromarray(raw).resize((300, 460))),\n                           np.asarray(Image.fromarray(window).resize((300, 460)))))\n        print(\'GMIC RSNA TECHNICAL IMAGE\', json.dumps(record), flush=True)\n    for start in range(0, len(records), 4):\n        fig, axes = plt.subplots(4, 2, figsize=(8, 16))\n        for offset, pair in enumerate(axes):\n            idx = start+offset\n            for ax in pair: ax.axis(\'off\')\n            if idx >= len(records): continue\n            item = records[idx]\n            for col, image in enumerate(thumbnails[idx]):\n                pair[col].imshow(image, cmap=\'gray\')\n                pair[col].set_title(f"Sample {idx+1} site {item[\'site_id\']} {item[\'view\']} | "\n                                    + (\'native presentation\' if col == 0 else \'GMIC window\'))\n        fig.suptitle(\'Private technical orientation/crop review; no diagnostic interpretation\')\n        fig.tight_layout()\n        fig.savefig(ROOT/f\'contact_sheet_{start//4+1:02d}_private.png\', dpi=100)\n        plt.close(fig)\n    result = {\'images\': len(records), \'sampling_partition\': \'pilot_fit\', \'records\': records,\n              \'visual_review_status\': \'pending\', \'full_cohort_transfer_evaluation_authorized_by_gate\': False}\n    write_json(ROOT/\'rsna_technical_summary.json\', result)\n    return result\n\n\ndef main():\n    ROOT.mkdir(parents=True, exist_ok=True)\n    require(torch.cuda.is_available(), \'Kaggle GPU required.\')\n    require(source_fingerprint() == \'f4178761b59233ffe1048218d234624c22764c8129d11c48fe30821cbe079a7c\',\n            \'Original model package changed.\')\n    torch.manual_seed(17)\n    np.random.seed(17)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    declared = {**SETTINGS, \'environment\': environment(),\n                \'script_sha256\': digest_file(__file__),\n                \'adapter_sha256\': digest_file(\'gmic_adapter.py\'),\n                \'manifest_sha256\': digest_file(\'gmic_upstream_manifest.json\')}\n    write_json(ROOT/\'declared_plan.json\', declared)\n    print(\'GMIC DECLARED PLAN\', json.dumps(declared), flush=True)\n    upstream = ROOT/\'upstream\'\n    files = acquire_upstream(\'gmic_upstream_manifest.json\', upstream)\n    write_json(ROOT/\'verified_upstream_files.json\', files)\n    sys.path.insert(0, str(upstream))\n    models = load_models(upstream)\n    reference = reproduce_reference(upstream, models)\n    technical = check_rsna(models)\n    result = {\'status\': \'numeric_preflight_completed_visual_review_pending\',\n              \'reference_compatibility_passed\': reference[\'compatibility_passed\'],\n              \'current_fixture_vs_readme_passed\': reference[\'passed\'],\n              \'current_fixture_vs_readme_max_error\': reference[\'max_absolute_error\'],\n              \'historical_reference_max_error\': reference.get(\'historical_reference_max_error\'),\n              \'current_cli_max_error\': reference.get(\'current_cli_max_error\'),\n              \'rsna_fit_images_checked\': technical[\'images\'], \'final_test_results_opened\': False,\n              \'rsna_accuracy_estimated\': False, \'visual_review_required\': True,\n              \'upstream_commit\': UPSTREAM_COMMIT,\n              \'peak_gpu_allocated_gib\': torch.cuda.max_memory_allocated()/1024**3}\n    write_json(ROOT/\'completed.json\', result)\n    print(\'GMIC PREFLIGHT COMPLETED\', json.dumps(result), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
(PROJECT / 'gmic_fullfield.py').write_text('"""RSNA full-field adaptation for fixed GMIC weights; not NYU preprocessing."""\nimport numpy as np\nfrom PIL import Image\nfrom gmic_adapter import infer_horizontal_flip\n\n\ndef fullfield_array(raw, view, height=2944, width=1920):\n    """Orient, fit the complete image extent, and pad without a crop window.\n\n    Larger images are downsampled with preserved aspect ratio; smaller images\n    are not enlarged. This avoids geometric exclusion but cannot guarantee\n    preservation of small diagnostic details. No label enters this function.\n    """\n    if min(height, width) <= 0:\n        raise ValueError(\'Invalid target dimensions.\')\n    flip_flag, orientation = infer_horizontal_flip(raw, view)\n    flip = ((flip_flag == \'NO\' and view.startswith(\'R-\')) or\n            (flip_flag == \'YES\' and view.startswith(\'L-\')))\n    oriented = np.fliplr(raw) if flip else raw\n    h, w = oriented.shape\n    scale = min(1., height/h, width/w)\n    new_h, new_w = max(1, round(h*scale)), max(1, round(w*scale))\n    resized = np.asarray(Image.fromarray(oriented.astype(np.float32)).resize(\n        (new_w, new_h), resample=Image.Resampling.BILINEAR))\n    canvas = np.zeros((height, width), dtype=np.float32)\n    # Chest wall on the left. Match the released CC/MLO padding convention.\n    top = (height-new_h)//2 if view.endswith(\'CC\') else 0\n    canvas[top:top+new_h, :new_w] = resized\n    processed = canvas.copy()\n    processed -= processed.mean()\n    processed /= max(float(processed.std()), 1e-5)\n    if not np.isfinite(processed).all() or processed.std() < .99:\n        raise ValueError(\'Invalid standardized full-field input.\')\n    details = {**orientation, \'horizontal_flip\': flip_flag, \'actual_flip_applied\': flip,\n               \'raw_shape\': [h,w], \'resized_shape\': [new_h,new_w],\n               \'canvas_shape\': [height,width], \'scale\': float(scale),\n               \'padding_top\': top, \'padding_left\': 0,\n               \'source_window\': [0,h,0,w], \'geometric_cropping\': False,\n               \'normalized_mean\': float(processed.mean()), \'normalized_std\': float(processed.std())}\n    return processed, canvas, details\n')
(PROJECT / 'gmic_fullfield_preflight.py').write_text('"""Audit full-field GMIC adaptation on the exact original 20 fit images."""\nfrom pathlib import Path\nimport json\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\n\nfrom gmic_adapter import digest_file, decode_native, filter_legacy_state, UPSTREAM_COMMIT\nfrom gmic_fullfield import fullfield_array\nfrom gmic_preflight import infer\nfrom mammo_reliability.audit import locate_dataset, require, write_json, environment\n\nOUTPUT = Path(\'/kaggle/working/gmic_fullfield_preflight\')\nEXPECTED_PLAN = \'6f0aff552657aebfe4a4da73ddeacfdf29e62daaa3b590c34e7ef499949fdfc2\'\n\n\ndef verified_preflight():\n    candidates = [Path(\'/kaggle/input/notebooks/aseqwss/09-rsna-gmic-transfer-preflight/gmic_transfer_preflight\'),\n                  Path(\'/kaggle/input/09-rsna-gmic-transfer-preflight/gmic_transfer_preflight\')]\n    existing = [x for x in candidates if x.is_dir()]\n    require(len(existing) == 1, \'Attach private notebook 09 version 4.\')\n    base = existing[0]\n    require(digest_file(base/\'declared_plan.json\') == EXPECTED_PLAN, \'Wrong preflight version.\')\n    result = json.loads((base/\'completed.json\').read_text())\n    require(result[\'reference_compatibility_passed\'], \'Compatibility checks did not pass.\')\n    require(result[\'rsna_fit_images_checked\'] == 20 and not result[\'final_test_results_opened\'], \'Unexpected preflight.\')\n    manifest = json.loads(Path(\'gmic_upstream_manifest.json\').read_text())\n    require(manifest[\'commit\'] == UPSTREAM_COMMIT, \'Changed upstream source.\')\n    upstream = base/\'upstream\'\n    for name, blob in manifest[\'files\'].items():\n        require(digest_file(upstream/name, \'sha1\', git_blob=True) == blob, f\'Upstream file changed: {name}\')\n    return base, upstream\n\n\ndef load_fixed_models(upstream, output):\n    sys.path.insert(0, str(upstream))\n    from src.modeling.gmic import GMIC\n    from src.constants import PERCENT_T_DICT\n    models, records = [], []\n    for index in range(1,6):\n        params = {\'device_type\':\'gpu\',\'gpu_number\':0,\'cam_size\':(46,30),\'K\':6,\n                  \'crop_shape\':(256,256),\'post_processing_dim\':256,\'num_classes\':2,\n                  \'use_v1_global\':False,\'percent_t\':PERCENT_T_DICT[str(index)]}\n        model = GMIC(params)\n        checkpoint = upstream/\'models\'/f\'sample_model_{index}.p\'\n        state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n        active, ignored = filter_legacy_state(state, model.state_dict().keys())\n        model.load_state_dict(active, strict=True)\n        model.eval().to(\'cuda\')\n        for parameter in model.parameters(): parameter.requires_grad_(False)\n        models.append(model)\n        records.append({\'model\':index,\'checkpoint_sha256\':digest_file(checkpoint),\'ignored_unused_keys\':ignored})\n    write_json(output/\'fixed_models.json\',records)\n    return models\n\n\ndef main():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    OUTPUT.mkdir(parents=True,exist_ok=True)\n    require(torch.cuda.is_available(), \'GPU required.\')\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    base, upstream = verified_preflight()\n    roles_path=base/\'cohort_identity/pilot/roles_private.csv\'\n    require(digest_file(roles_path)==\'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\',\'Roles changed.\')\n    roles=pd.read_csv(roles_path)\n    fit_ids=set(roles.loc[roles.role.eq(\'fit\'),\'patient_id\'])\n    sample=pd.read_csv(base/\'engineering_sample_private.csv\')\n    require(len(sample)==20 and sample.image_id.is_unique and set(sample.patient_id).issubset(fit_ids), \'Wrong engineering sample.\')\n    declaration={\'purpose\':\'full_field_geometry_repair_on_same_20_fit_images\',\'preflight_plan_sha256\':EXPECTED_PLAN,\n                 \'preprocessing\':\'full original field, horizontal orientation heuristic, downscale only to fit, left-aligned padding\',\n                 \'target_shape\':[2944,1920],\'interpolation\':\'bilinear\',\'image_extent_cropped\':False,\n                 \'caution\':\'Downsampling can lose fine detail; this is an explicit RSNA adaptation, not exact NYU preprocessing.\',\n                 \'frozen_models\':5,\'accuracy_calculated\':False,\'final_test_results_opened\':False,\n                 \'environment\':environment(),\'script_sha256\':digest_file(__file__),\n                 \'fullfield_sha256\':digest_file(\'gmic_fullfield.py\'),\'adapter_sha256\':digest_file(\'gmic_adapter.py\')}\n    write_json(OUTPUT/\'declared_plan.json\',declaration)\n    print(\'FULLFIELD DECLARED PLAN\',json.dumps(declaration),flush=True)\n    models=load_fixed_models(upstream,OUTPUT)\n    data_root=locate_dataset()\n    require(digest_file(data_root/\'train.csv\')==\'35408731b67b87272f85874ee3c4c9d5cea7a2c4fbcaeac3fbe415ba1a5817c1\',\'Metadata changed.\')\n    records,thumbs=[],[]\n    for i,row in enumerate(sample.itertuples(index=False),1):\n        started=time.monotonic()\n        raw,header=decode_native(data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\')\n        view=f\'{row.laterality}-{row.view}\'\n        array,canvas,geometry=fullfield_array(raw,view)\n        predictions=infer(models,array)\n        np.save(OUTPUT/f\'sample_{i:02d}_predictions_private.npy\',predictions)\n        records.append({\'sample\':i,\'site_id\':int(row.site_id),\'machine_id\':int(row.machine_id),\n                        \'view\':view,**header,**geometry,\'seconds\':time.monotonic()-started})\n        pair=[]\n        for img in [raw,canvas]:\n            # Pillow cannot reduce every uint16 mode; float preserves display range.\n            thumb=Image.fromarray(img.astype(np.float32));thumb.thumbnail((500,500));pair.append(np.asarray(thumb))\n        thumbs.append(pair)\n        print(\'FULLFIELD IMAGE\',json.dumps(records[-1]),flush=True)\n    for start in range(0,20,4):\n        fig,axes=plt.subplots(4,2,figsize=(10,16),layout=\'constrained\')\n        for j in range(4):\n            i=start+j\n            for col in range(2):\n                axes[j,col].imshow(thumbs[i][col],cmap=\'gray\',aspect=\'equal\')\n                axes[j,col].axis(\'off\')\n                axes[j,col].set_title(f"Sample {i+1}, site {records[i][\'site_id\']}, {records[i][\'view\']}\\n"\n                                      +(\'Original presentation\' if col==0 else \'Full-field model input\'),fontsize=10)\n        fig.suptitle(\'Private engineering review; no lesion or cancer interpretation\')\n        fig.savefig(OUTPUT/f\'contact_sheet_{start//4+1:02d}_private.png\',dpi=100)\n        plt.close(fig)\n    write_json(OUTPUT/\'technical_images.json\',records)\n    result={\'status\':\'numeric_fullfield_preflight_completed_visual_review_pending\',\'images\':20,\n            \'final_test_results_opened\':False,\'accuracy_calculated\':False,\'visual_review_required\':True,\n            \'minimum_scale\':min(r[\'scale\'] for r in records),\'maximum_scale\':max(r[\'scale\'] for r in records),\n            \'peak_gpu_allocated_gib\':torch.cuda.max_memory_allocated()/1024**3,\n            \'declared_plan_sha256\':digest_file(OUTPUT/\'declared_plan.json\')}\n    write_json(OUTPUT/\'completed.json\',result)\n    print(\'FULLFIELD COMPLETED\',json.dumps(result),flush=True)\n\n\nif __name__==\'__main__\': main()\n')
(PROJECT / 'gmic_pilot_analysis.py').write_text('"""Explicit probability pooling for fixed external GMIC development models.\n\nThis is separate from the original logit-pooling ResNet pipeline. It does not\ncalibrate, choose a threshold, select a member, or estimate a final endpoint.\n"""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom mammo_reliability.audit import BREAST_KEY, require\n\nMEMBERS = list(range(1, 6))\nPREDICTION_COLUMNS = [f\'m{m}_{target}\' for m in MEMBERS for target in [\'benign\', \'malignant\']]\n\n\ndef aggregate_predictions(images, predictions):\n    """Average repeats within view, then give CC and MLO equal weight."""\n    require(images.image_id.is_unique and predictions.image_id.is_unique, \'Duplicate image identifiers.\')\n    require(set(images.image_id) == set(predictions.image_id), \'Incomplete or unexpected prediction coverage.\')\n    require(set(PREDICTION_COLUMNS).issubset(predictions), \'Missing model output columns.\')\n    values = predictions[PREDICTION_COLUMNS].to_numpy(dtype=float)\n    require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(), \'Invalid predictions.\')\n    require(images.view.isin([\'CC\', \'MLO\']).all(), \'Nonstandard view in the pilot.\')\n    require(images.role.isin([\'fit\', \'tune\']).all(), \'A nondevelopment role entered the pilot.\')\n    require(images.cancer.isin([0, 1]).all(), \'Invalid cancer labels.\')\n    groups = images.groupby(BREAST_KEY)\n    require((groups.view.nunique() == 2).all(), \'A breast lacks a standard view.\')\n    for column in [\'cancer\', \'site_id\', \'role\']:\n        require((groups[column].nunique(dropna=False) == 1).all(), f\'Conflicting {column}.\')\n    require((images.groupby(\'patient_id\').role.nunique() == 1).all(), \'Patient crosses pilot roles.\')\n    require(not set(PREDICTION_COLUMNS).intersection(images.columns), \'Predictions already in metadata.\')\n    joined = images.merge(predictions[[\'image_id\'] + PREDICTION_COLUMNS], on=\'image_id\', validate=\'one_to_one\')\n    malignant = [f\'m{m}_malignant\' for m in MEMBERS]\n    views = joined.groupby(BREAST_KEY + [\'view\'])[malignant].mean()\n    cc, mlo = views.xs(\'CC\', level=\'view\'), views.xs(\'MLO\', level=\'view\')\n    require(cc.index.equals(mlo.index), \'CC and MLO breast identities do not match.\')\n    result = groups.agg(site_id=(\'site_id\', \'first\'), role=(\'role\', \'first\'),\n                        cancer=(\'cancer\', \'first\'), n_images=(\'image_id\', \'size\'))\n    for member, column in zip(MEMBERS, malignant):\n        result[f\'m{member}_cc\'] = cc[column]\n        result[f\'m{member}_mlo\'] = mlo[column]\n        result[f\'m{member}_breast\'] = (cc[column] + mlo[column]) / 2\n    members = result[[f\'m{m}_breast\' for m in MEMBERS]].to_numpy()\n    result[\'p_raw\'] = members.mean(axis=1)\n    result[\'p_cc\'] = cc.mean(axis=1)\n    result[\'p_mlo\'] = mlo.mean(axis=1)\n    result[\'view_disagreement\'] = abs(result.p_cc - result.p_mlo)\n    result[\'ensemble_variance\'] = members.var(axis=1, ddof=0)\n    require(np.isfinite(result.p_raw).all(), \'Missing pooled predictions.\')\n    return result.reset_index()\n\n\ndef rank_metrics(frame, column):\n    y, p = frame.cancer.to_numpy(), frame[column].to_numpy()\n    return {\'breasts\': len(frame), \'positive_breasts\': int(y.sum()),\n            \'prevalence\': float(y.mean()),\n            \'average_precision\': float(average_precision_score(y, p)) if y.sum() else None,\n            \'roc_auc\': float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None}\n\n\ndef summarize_development(breasts):\n    rows = []\n    for role, group in breasts.groupby(\'role\', sort=True):\n        # All fixed members are reported; no best-member selection is performed.\n        for name, column in [(\'ensemble\', \'p_raw\')] + [(f\'model_{m}\', f\'m{m}_breast\') for m in MEMBERS]:\n            rows.append({\'role\': role, \'model\': name, \'site\': \'pooled\', **rank_metrics(group, column)})\n        for site, subset in group.groupby(\'site_id\'):\n            rows.append({\'role\': role, \'model\': \'ensemble\', \'site\': int(site), **rank_metrics(subset, \'p_raw\')})\n    return {\'purpose\': \'adaptive_reserved_development_only\', \'model_fitted_on_rsna\': False,\n            \'threshold_selected\': False, \'calibration_fitted\': False,\n            \'final_test_results_opened\': False, \'metrics\': rows,\n            \'interpretation\': \'Released RSNA labels; development estimates after prior adaptive trials. Not final performance.\'}\n')
(PROJECT / 'gmic_pilot.py').write_text('"""Fixed external mammography ensemble; reserved development patients only."""\nfrom pathlib import Path\nimport csv\nimport json\nimport os\nimport shutil\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom gmic_adapter import decode_native, digest_file\nfrom gmic_fullfield import fullfield_array\nfrom gmic_preflight import infer\nfrom gmic_fullfield_preflight import verified_preflight, load_fixed_models\nfrom gmic_pilot_analysis import aggregate_predictions, summarize_development, PREDICTION_COLUMNS\nfrom mammo_reliability.audit import locate_dataset, require, write_json, environment\nfrom mammo_reliability.training import source_fingerprint\n\nOUTPUT = Path(\'/kaggle/working/gmic_reserved_pilot\')\nEXPECTED_ROLES = \'fc0308c63300be6849c7ac9b830d09480c2b62d64df4f1d036bab12548a7e15f\'\nEXPECTED_METADATA = \'35408731b67b87272f85874ee3c4c9d5cea7a2c4fbcaeac3fbe415ba1a5817c1\'\n\n\ndef verified_geometry():\n    candidates = [Path(\'/kaggle/input/notebooks/aseqwss/10-rsna-gmic-fullfield-check/gmic_fullfield_preflight\'),\n                  Path(\'/kaggle/input/10-rsna-gmic-fullfield-check/gmic_fullfield_preflight\')]\n    existing = [p for p in candidates if p.is_dir()]\n    require(len(existing) == 1, \'Attach notebook 10 with the reviewed geometry audit.\')\n    root = existing[0]\n    review = json.loads(Path(\'gmic_visual_review.json\').read_text())\n    require(review[\'engineering_visual_review_passed\'] is True and review[\'images_reviewed\'] == 20,\n            \'Complete technical visual review first.\')\n    require(digest_file(root/\'declared_plan.json\') == review[\'declared_plan_sha256\'], \'Unreviewed audit version.\')\n    complete = json.loads((root/\'completed.json\').read_text())\n    require(complete[\'images\'] == 20 and not complete[\'final_test_results_opened\'], \'Incomplete image audit.\')\n    declaration = json.loads((root/\'declared_plan.json\').read_text())\n    for key, name in [(\'fullfield_sha256\', \'gmic_fullfield.py\'), (\'adapter_sha256\', \'gmic_adapter.py\')]:\n        require(declaration[key] == digest_file(name), f\'Unreviewed processing changes: {name}\')\n    return root, review\n\n\ndef main():\n    started = time.monotonic()\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    # A new run must never accidentally combine partial outputs with new code.\n    require(not (OUTPUT/\'image_predictions_private.csv\').exists(), \'Existing predictions require explicit verified recovery.\')\n    require(torch.cuda.is_available(), \'GPU required.\')\n    require(source_fingerprint() == \'f4178761b59233ffe1048218d234624c22764c8129d11c48fe30821cbe079a7c\',\n            \'Original model package changed.\')\n    torch.manual_seed(17)\n    np.random.seed(17)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    base, upstream = verified_preflight()\n    geometry_root, review = verified_geometry()\n    roles_path = base/\'cohort_identity/pilot/roles_private.csv\'\n    require(digest_file(roles_path) == EXPECTED_ROLES, \'Patient roles changed.\')\n    roles = pd.read_csv(roles_path)\n    require(roles.patient_id.is_unique and set(roles.cohort) == {\'pilot\'} and (roles.outer_fold == -1).all(),\n            \'A final evaluation patient entered development.\')\n    require(set(roles.role) == {\'fit\', \'tune\'} and len(roles) == 2383, \'Wrong development roles.\')\n    data_root = locate_dataset()\n    require(digest_file(data_root/\'train.csv\') == EXPECTED_METADATA, \'Metadata changed.\')\n    all_images = pd.read_csv(data_root/\'train.csv\')\n    images = all_images.loc[all_images.patient_id.isin(roles.patient_id) & all_images.view.isin([\'CC\', \'MLO\'])]\n    images = images.merge(roles[[\'patient_id\', \'role\']], on=\'patient_id\', validate=\'many_to_one\')\n    images = images.sort_values([\'patient_id\', \'laterality\', \'view\', \'image_id\']).reset_index(drop=True)\n    require(len(images) == 10881 and images.image_id.is_unique, \'Wrong pilot image coverage.\')\n    require(set(images.patient_id) == set(roles.patient_id), \'Missing development patients.\')\n    require(len(images[[\'patient_id\', \'laterality\']].drop_duplicates()) == 4766, \'Wrong breast count.\')\n    # The labels below are used only in the final descriptive analysis.\n    images.to_csv(OUTPUT/\'images_private.csv\', index=False)\n    shutil.copyfile(roles_path, OUTPUT/\'roles_private.csv\')\n    shutil.copyfile(\'gmic_visual_review.json\', OUTPUT/\'visual_review.json\')\n    declaration = {\n        \'purpose\': \'reserved_development_transfer_feasibility\', \'rsna_model_training\': False,\n        \'patients\': 2383, \'breasts\': 4766, \'images\': 10881,\n        \'roles_sha256\': EXPECTED_ROLES, \'metadata_sha256\': EXPECTED_METADATA,\n        \'upstream_commit\': \'594029efe6184bcca3a151fa1dd46c5fd620cb25\',\n        \'models\': [1, 2, 3, 4, 5], \'class_order\': [\'benign\', \'malignant\'],\n        \'inference\': \'FP32, one deterministic full-field image, fixed external weights, no TTA\',\n        \'pooling\': \'arithmetic malignant probability mean within repeated view; equal CC/MLO mean; equal five-model mean\',\n        \'uncertainty_descriptors\': \'absolute ensemble CC-MLO difference; population variance of five breast probabilities\',\n        \'analysis\': \'AP and ROC-AUC for all fixed members and ensemble by pilot role; ensemble also by site\',\n        \'threshold_selection\': False, \'calibration\': False, \'density_effect_testing\': False,\n        \'final_test_results_opened\': False, \'model_or_member_selection_in_job\': False,\n        \'orientation_low_contrast_flag\': .2, \'orientation_flags_exclude_images\': False,\n        \'environment\': environment(), \'gpu\': torch.cuda.get_device_name(0),\n        \'source_sha256\': {p: digest_file(p) for p in [\n            \'gmic_pilot.py\', \'gmic_pilot_analysis.py\', \'gmic_adapter.py\', \'gmic_preflight.py\',\n            \'gmic_fullfield.py\', \'gmic_fullfield_preflight.py\', \'gmic_visual_review.json\']},\n        \'geometry_review\': review,\n        \'limitations\': [\'Adapted image scale can reduce small-feature visibility.\',\n                       \'Heuristic orientation is not validated for every image.\',\n                       \'Five published models were selected in external hyperparameter searches, not a seed-only ensemble.\',\n                       \'Different reported source institutions do not prove patient-level independence.\',\n                       \'Adaptive development estimates are not final research results.\']}\n    write_json(OUTPUT/\'declared_plan.json\', declaration)\n    print(\'GMIC PILOT DECLARED PLAN\', json.dumps(declaration), flush=True)\n    models = load_fixed_models(upstream, OUTPUT)\n    # Reproduce every model on the image that exposed the earlier crop failure.\n    # A pandas Series has a .view method; use a named row for the view column.\n    sample = next(pd.read_csv(base/\'engineering_sample_private.csv\').iloc[[10]].itertuples(index=False))\n    require(sample.patient_id in set(roles.loc[roles.role.eq(\'fit\'), \'patient_id\']), \'Probe is not a fitting image.\')\n    raw, _ = decode_native(data_root/\'train_images\'/str(sample.patient_id)/f\'{sample.image_id}.dcm\')\n    array, _, _ = fullfield_array(raw, f\'{sample.laterality}-{sample.view}\')\n    probe = infer(models, array)\n    expected = np.load(geometry_root/\'sample_11_predictions_private.npy\', allow_pickle=False)\n    error = float(np.max(abs(probe-expected)))\n    require(error <= 1e-6, \'The reviewed image predictions did not reproduce.\')\n    write_json(OUTPUT/\'reproduction_probe.json\', {\'sample\': 11, \'max_absolute_error\': error, \'tolerance\': 1e-6})\n    print(\'GMIC PILOT REPRODUCTION\', json.dumps({\'sample\': 11, \'max_absolute_error\': error}), flush=True)\n    fields = [\'image_id\'] + PREDICTION_COLUMNS + [\'scale\', \'edge_mass_contrast\', \'actual_flip_applied\',\n                                                \'raw_height\', \'raw_width\', \'native_file_sha256\']\n    with (OUTPUT/\'image_predictions_private.csv\').open(\'w\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for count, row in enumerate(images.itertuples(index=False), 1):\n            path = data_root/\'train_images\'/str(row.patient_id)/f\'{row.image_id}.dcm\'\n            raw, _ = decode_native(path)\n            array, _, geometry = fullfield_array(raw, f\'{row.laterality}-{row.view}\')\n            predictions = infer(models, array)\n            record = {\'image_id\': row.image_id, **dict(zip(PREDICTION_COLUMNS, predictions.ravel().tolist())),\n                      \'scale\': geometry[\'scale\'], \'edge_mass_contrast\': geometry[\'edge_mass_contrast\'],\n                      \'actual_flip_applied\': geometry[\'actual_flip_applied\'],\n                      \'raw_height\': raw.shape[0], \'raw_width\': raw.shape[1], \'native_file_sha256\': digest_file(path)}\n            writer.writerow(record)\n            if count % 100 == 0 or count == len(images):\n                stream.flush(); os.fsync(stream.fileno())\n                progress = {\'images_completed\': count, \'images_total\': len(images),\n                            \'elapsed_seconds\': time.monotonic()-started, \'final_test_results_opened\': False}\n                write_json(OUTPUT/\'progress.json\', progress)\n                print(\'GMIC PILOT PROGRESS\', json.dumps(progress), flush=True)\n    predictions = pd.read_csv(OUTPUT/\'image_predictions_private.csv\')\n    breasts = aggregate_predictions(images, predictions)\n    require(len(breasts) == 4766 and int(breasts.cancer.sum()) == 99, \'Wrong outcome coverage.\')\n    require(breasts.groupby(\'role\').size().to_dict() == {\'fit\': 3574, \'tune\': 1192}, \'Wrong role coverage.\')\n    breasts.to_csv(OUTPUT/\'breast_predictions_private.csv\', index=False)\n    summary = summarize_development(breasts)\n    summary[\'geometry\'] = {\'minimum_scale\': float(predictions.scale.min()),\n                           \'maximum_scale\': float(predictions.scale.max()),\n                           \'low_orientation_contrast_images\': int((predictions.edge_mass_contrast < .2).sum()),\n                           \'minimum_edge_mass_contrast\': float(predictions.edge_mass_contrast.min())}\n    write_json(OUTPUT/\'development_summary.json\', summary)\n    for record in summary[\'metrics\']: print(\'GMIC PILOT METRICS\', json.dumps(record), flush=True)\n    result = {\'status\': \'reserved_development_inference_completed\', \'images\': len(images), \'breasts\': len(breasts),\n              \'elapsed_seconds\': time.monotonic()-started, \'final_test_results_opened\': False,\n              \'declared_plan_sha256\': digest_file(OUTPUT/\'declared_plan.json\'),\n              \'image_predictions_sha256\': digest_file(OUTPUT/\'image_predictions_private.csv\'),\n              \'breast_predictions_sha256\': digest_file(OUTPUT/\'breast_predictions_private.csv\'),\n              \'summary_sha256\': digest_file(OUTPUT/\'development_summary.json\')}\n    write_json(OUTPUT/\'completed.json\', result)\n    print(\'GMIC PILOT COMPLETED\', json.dumps(result), flush=True)\n\n\nif __name__ == \'__main__\': main()\n')
(PROJECT / 'gmic_upstream_manifest.json').write_text('{\n  "repository": "https://github.com/nyukat/GMIC",\n  "commit": "594029efe6184bcca3a151fa1dd46c5fd620cb25",\n  "observed_date": "2026-09-24",\n  "license": "AGPL-3.0; preserve upstream LICENSE and NOTICE",\n  "files": {\n    "LICENSE": "29ebfa545f5580919a4e884d7014d7a3eb2df762",\n    "NOTICE": "8306b2920b99173f2f0d0aead128cbe66701dc11",\n    "README.md": "b008e2dfb4cfec1519179af41dc53ebbe1a386bc",\n    "models/sample_model_1.p": "04e473379a8313daa426c38e69bad392fc0be769",\n    "models/sample_model_2.p": "98074a24bbebdbf030eb2bd9c27e96f4683d336b",\n    "models/sample_model_3.p": "9304d4b3224ecd4b9bd6717095ef819c779efae5",\n    "models/sample_model_4.p": "4e87f6453d71e9e9be01504ee8b64ff077c254ee",\n    "models/sample_model_5.p": "ec942dd65a99aaa3fce8cf00bcd10bdb0ee40e2c",\n    "run.sh": "d1fa003036b64b4a535ae6e470196e785389fc53",\n    "sample_data/exam_list_before_cropping.pkl": "f597ecd9625e0f5dbda1d61f32639a9f6db9bfb8",\n    "sample_data/images/0_L-CC.png": "d56dcb335be44a3caf433676ea42a4daaf4fc9d1",\n    "sample_data/images/0_L-MLO.png": "8705440c77a2b6f7658ce0e13c65cabde6c08fa2",\n    "sample_data/images/0_R-CC.png": "1c224db36fcaf44bf95997e39fec9de64ac93754",\n    "sample_data/images/0_R-MLO.png": "9db231fa8a57bf38cc80b0a51f9a1335b9fd9096",\n    "sample_data/images/1_L-CC.png": "81a5447498e89969a566d007442923111cde8f65",\n    "sample_data/images/1_L-MLO.png": "a233ae4588344374fb28c3e5985479cf7525d37b",\n    "sample_data/images/1_R-CC.png": "caba1c1cc4fa1da9d14740579395a0f884e5343c",\n    "sample_data/images/1_R-MLO.png": "dcdec558d214075769a01580aac1c9865e2a5bf2",\n    "sample_data/images/2_L-CC.png": "5ed0ea342155c629da1f62bb1cc177bfdb9f8dfe",\n    "sample_data/images/2_L-MLO.png": "f742384c1702c1f1516cbf54ea5e282af0d48f19",\n    "sample_data/images/2_R-CC.png": "9b1d544812941ca03b0545123aaf1e4867ecea2f",\n    "sample_data/images/2_R-MLO.png": "7e4785aadb19874fa7ff5fe64903181a7a33eb86",\n    "sample_data/images/3_L-CC.png": "6c7e07d0be539aa1f4c0c4dbb0e9a3e2fb15a161",\n    "sample_data/images/3_L-MLO.png": "716d35236bbce8ed438dc6e80c54faaccbd6acd7",\n    "sample_data/images/3_R-CC.png": "c4f33f30da14fbc19ba9d9a4adcd008797e36480",\n    "sample_data/images/3_R-MLO.png": "739279f1ba23ee568da73f95661539aea808f33e",\n    "src/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/constants.py": "52b78212e611fe9347f2d96b97c9be8d616f1b32",\n    "src/cropping/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/cropping/crop_mammogram.py": "0afb623e3356206d66c649a144f6fd5609e88feb",\n    "src/cropping/crop_single.py": "f57b90ed8db356e96861324e405ad51de40eee37",\n    "src/data_loading/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/data_loading/augmentations.py": "143a4eb4ec43559a0a9fa07788db3dfafe1c9a63",\n    "src/data_loading/loading.py": "94bbd001a4e3b263b0e991319bc2fa59bb25c04f",\n    "src/modeling/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/modeling/gmic.py": "2c3a615a78546d5923a39624732eb799325218e5",\n    "src/modeling/modules.py": "729dd9a3659d2ec3b4ef89be931cd3d6097c8684",\n    "src/optimal_centers/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/optimal_centers/calc_optimal_centers.py": "3a5057669dd625f82ef830ec96b4d88c468c88d5",\n    "src/optimal_centers/get_optimal_center_single.py": "5030d2fbcd8c2826bcba339d3e9e379f9b8228aa",\n    "src/optimal_centers/get_optimal_centers.py": "2e855972b1fb9da8660a1039f619b5791df0824e",\n    "src/scripts/run_model.py": "9024a52ed004805659aa944609d5051987b3de94",\n    "src/utilities/__init__.py": "e69de29bb2d1d6434b8b29ae775ad8c2e48c5391",\n    "src/utilities/data_handling.py": "483aeb112789d6feed96d9a363c413836a105f6c",\n    "src/utilities/pickling.py": "01814cce7598956fc829312b22e4fd890ae74e39",\n    "src/utilities/reading_images.py": "754a04487f6616316f12faf02d84a9f1953e1013",\n    "src/utilities/saving_images.py": "8a7880a51c85fc8660e5f800fb79b32ca337bf26",\n    "src/utilities/tools.py": "676a10dcb70d94bb031c3b86213c6fe865814f56"\n  }\n}\n')
(PROJECT / 'gmic_historical_manifest.json').write_text('{\n  "commit": "febe2431dbe6efd75bcb37ddea85dc48c0df0b07",\n  "files": {\n    "README.md": "43f86e2104efae2321a7929c70423ac4c9deb842",\n    "run.sh": "2a61968fc6032255b439377eae529aac910b04ee",\n    "sample_data/cropped_images/0_L-CC.png": "7cd256cf1e963605365c8b57ebbb2860bd671b8e",\n    "sample_data/cropped_images/0_L-MLO.png": "45a7fed5e3aba7ae20fbcc33dea80e8ce8b98eac",\n    "sample_data/cropped_images/0_R-CC.png": "fa99f5e2e82177e32b5ba8ef2463e25e83015c0a",\n    "sample_data/cropped_images/0_R-MLO.png": "1be35c719c2cd1343181e2c8b2851906d82670ce",\n    "sample_data/cropped_images/1_L-CC.png": "468a9c77e591b70017055331677b2361091eaec2",\n    "sample_data/cropped_images/1_L-MLO.png": "df60bef584646edc2b21fb5100ec28f82938895f",\n    "sample_data/cropped_images/1_R-CC.png": "2d090efed2ac4ad53a426401406546d45d5ae346",\n    "sample_data/cropped_images/1_R-MLO.png": "c0e3dccfb3df4020b64be063bdb9b32a3d8cb277",\n    "sample_data/cropped_images/2_L-CC.png": "b598f6202f063494cbf0963f431fc5351ec55a8c",\n    "sample_data/cropped_images/2_L-MLO.png": "b1d46a6b13084bbcd97b08add7b8f50cf06e95e3",\n    "sample_data/cropped_images/2_R-CC.png": "fd92be32967a137a2168837a9b11e386b0c93f41",\n    "sample_data/cropped_images/2_R-MLO.png": "211f3c7f84de57a03e16f278f4abefcf3e55b982",\n    "sample_data/cropped_images/3_L-CC.png": "409d495d362517a4a6168ad7a047a18de0886b52",\n    "sample_data/cropped_images/3_L-MLO.png": "14e81fe755c968e5a017bfe4fc65e27a4a2ec441",\n    "sample_data/cropped_images/3_R-CC.png": "77508ab3494be0caa840592a14d25f66780cca14",\n    "sample_data/cropped_images/3_R-MLO.png": "fbb67d1506cc2e4a95f344ae3c61d68d3d8399fa",\n    "sample_data/data.pkl": "c5cda8816c66d7d3dc9b55e7ea99ed0f7d645fc3",\n    "src/data_loading/loading.py": "9aba98e5c6fb6027003106fbb099919b118bbdb4",\n    "src/scripts/run_model.py": "d32a0f62fc4ea2b65c3bbe944985ae183fb0ac5c"\n  },\n  "repository": "https://github.com/nyukat/GMIC",\n  "purpose": "Historical already-windowed author fixtures for unchanged README table",\n  "checkpoint_blobs_identical_to_current_commit": true\n}\n')
(PROJECT / 'gmic_visual_review.json').write_text('{\n  "observed_date": "2026-09-24",\n  "notebook": "10_rsna_gmic_fullfield_check",\n  "version": 2,\n  "script_version_id": 352360882,\n  "declared_plan_sha256": "94c490d11b274dedc01998942e9b0251ef73fbcdffdcb6e8d8d502174b92d81e",\n  "reviewer": "AI assistant, technical visual inspection; no radiologist review",\n  "engineering_visual_review_passed": true,\n  "images_reviewed": 20,\n  "sample_partition": "same deterministic pilot fitting images used by notebook 09",\n  "contact_sheets_reviewed": [\n    "contact_sheet_01_private.png",\n    "contact_sheet_02_private.png",\n    "contact_sheet_03_private.png",\n    "contact_sheet_04_private.png",\n    "contact_sheet_05_private.png"\n  ],\n  "observations": [\n    "All 20 original/model-input pairs inspected at contact-sheet scale.",\n    "No visible full-image boundary exclusion, blank output, or gross grayscale inversion in these pairs.",\n    "Model inputs consistently present the chest wall on the left in these examples.",\n    "Sample 11 now retains the visible breast; the original fixed window was largely background.",\n    "Substantial background and small breast areas remain in some acquisitions, including samples 5 and 16."\n  ],\n  "limitations": [\n    "This is an engineering review of 20 images, not diagnostic or lesion-preservation validation.",\n    "Downscaling ranged from 0.3906408952187182 to 1; fine features can be lost.",\n    "Orientation is a heuristic; success on these examples does not validate the entire cohort.",\n    "No cancer predictions were judged and no RSNA cohort accuracy was used to select the repair."\n  ],\n  "allowed_next_stage": "reserved pilot transfer feasibility only",\n  "final_evaluation_unlocked": false\n}\n')
subprocess.run([sys.executable,'-u','gmic_pilot.py'],cwd=PROJECT,check=True)
